# 株価データ取得プログラム
このノートブックは、指定した銘柄の株価データを取得し、CSVファイルとしてダウンロードします。

## 1. 必要なライブラリのインストールと読み込み

In [ ]:
# 必要なライブラリをインストール
!pip install -q yfinance pandas matplotlib japanize-matplotlib

# ライブラリの読み込み
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from datetime import datetime, timedelta
from google.colab import files

try:
    import japanize_matplotlib
    print('✓ 日本語フォント設定完了')
except ImportError:
    print('⚠️ japanize-matplotlib が見つかりません。ラベルが英語になる場合があります。')


## 2. 株価データ取得の設定
ここで銘柄コード、期間を設定します

In [ ]:
# ===== ここを編集してください =====

# 銘柄コード（例）
# 日本株: "7203.T" (トヨタ自動車)、"6758.T" (ソニー)
# 米国株: "AAPL" (Apple)、"MSFT" (Microsoft)、"GOOGL" (Google)
ticker_symbol = "7203.T"  # ここに取得したい銘柄コードを入力

# 期間設定（デフォルト: 直近1年間）
_today = datetime.today()
end_date   = _today.strftime("%Y-%m-%d")              # 本日
start_date = (_today - timedelta(days=365)).strftime("%Y-%m-%d")  # 1年前

# ★ 期間を手動で指定したい場合は、下の2行のコメントを外して入力してください
# start_date = "2024-01-01"
# end_date   = "2025-01-01"

# データの間隔（"1d"=日次、"1wk"=週次、"1mo"=月次）
interval = "1d"

# ===== 入力値の検証 =====
import re

date_pattern = r'^\d{4}-\d{2}-\d{2}$'
if not re.match(date_pattern, start_date) or not re.match(date_pattern, end_date):
    raise ValueError("日付は YYYY-MM-DD 形式で入力してください")

try:
    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt   = datetime.strptime(end_date,   "%Y-%m-%d")
    if start_dt >= end_dt:
        raise ValueError("開始日は終了日より前の日付にしてください")
except ValueError as e:
    raise ValueError(f"日付が無効です: {e}")

if not ticker_symbol or ticker_symbol.strip() == "":
    raise ValueError("銘柄コードを入力してください")

valid_intervals = ["1m","2m","5m","15m","30m","60m","90m","1h","1d","5d","1wk","1mo","3mo"]
if interval not in valid_intervals:
    raise ValueError(f"データ間隔は次のいずれかを指定してください: {', '.join(valid_intervals)}")

print(f"銘柄コード: {ticker_symbol}")
print(f"期間: {start_date} から {end_date}")
print(f"データ間隔: {interval}")
print("✓ 入力値の検証が完了しました")


## 3. 株価データの取得

In [ ]:
# 株価データを取得
print(f"\n{ticker_symbol} のデータを取得中...")

# yfinanceを使用してデータ取得（エラーハンドリング追加）
try:
    stock_data = yf.download(
        ticker_symbol,
        start=start_date,
        end=end_date,
        interval=interval,
        progress=True
    )
except Exception as e:
    error_msg = f"❌ データ取得中にエラーが発生しました: {str(e)}\n"
    error_msg += "以下を確認してください:\n"
    error_msg += "  - 銘柄コードが正しいか\n"
    error_msg += "  - インターネット接続が正常か\n"
    error_msg += "  - 日付範囲が適切か"
    raise RuntimeError(error_msg)

# データが取得できたか確認（重要: 空の場合は処理を中断）
if stock_data.empty:
    error_msg = "⚠️ データが取得できませんでした。\n"
    error_msg += "以下を確認してください:\n"
    error_msg += f"  - 銘柄コード「{ticker_symbol}」が正しいか\n"
    error_msg += f"  - 期間「{start_date}」から「{end_date}」にデータが存在するか\n"
    error_msg += "  - 日本株の場合は末尾に「.T」が付いているか (例: 7203.T)\n"
    error_msg += "\n⚠️ これ以降のセルは実行しないでください（空のファイルがダウンロードされます）"
    raise ValueError(error_msg)

# データ取得成功
print(f"✓ データ取得完了: {len(stock_data)} 件のデータ")
print(f"✓ カラム数: {len(stock_data.columns)}")
print(f"✓ カラム名: {', '.join(map(str, stock_data.columns))}")
print("\n最初の5件:")
print(stock_data.head())
print("\n最後の5件:")
print(stock_data.tail())

## 4. データの整形と確認

In [ ]:
# データフレームの情報を表示
print("データの概要:")
print(stock_data.info())

print("\n基本統計量:")
print(stock_data.describe())

# カラム名を日本語に変更（オプション）
# 注意: yfinanceのカラム構造が変更された場合に対応
stock_data_jp = stock_data.copy()

# 日本語化フラグの初期化
columns_converted_to_japanese = False

# カラム数の検証
expected_columns = 6
if len(stock_data_jp.columns) != expected_columns:
    print(f"⚠️ 警告: 予期しないカラム数です（期待: {expected_columns}, 実際: {len(stock_data_jp.columns)}）")
    print(f"カラム名: {list(map(str, stock_data_jp.columns))}")
    print("日本語カラム名への変換をスキップします。")
else:
    # 期待されるカラム名を検証（文字列に変換して比較）
    expected_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    actual_cols = [str(col) for col in stock_data_jp.columns]
    
    if actual_cols == expected_cols:
        stock_data_jp.columns = ['始値', '高値', '安値', '終値', '調整後終値', '出来高']
        columns_converted_to_japanese = True  # フラグを設定
        print("✓ カラム名を日本語に変換しました")
        print("\n日本語カラム名でのプレビュー:")
        print(stock_data_jp.head())
    else:
        print(f"⚠️ 警告: カラム名が期待と異なります")
        print(f"期待: {expected_cols}")
        print(f"実際: {actual_cols}")
        print("日本語カラム名への変換をスキップします。元のカラム名を使用してください。")

## 4-2. 株価チャート（テクニカル分析付き）

取得した株価データを3つのパネルで可視化します。
- **上段**: 終値 ＋ 移動平均線（SMA20/50/200）＋ ボリンジャーバンド（±2σ）
- **中段**: 出来高（上昇日=緑、下落日=赤）
- **下段**: RSI(14) — 70超=過熱圏、30未満=売られ過ぎ圏

In [ ]:
# ----- テクニカル指標の計算 -----
_df = stock_data.copy()
if isinstance(_df.columns, pd.MultiIndex):
    _df.columns = _df.columns.get_level_values(0)

close  = _df['Close'].squeeze().astype(float)
open_  = _df['Open'].squeeze().astype(float)
volume = _df['Volume'].squeeze().astype(float)
dates  = _df.index

sma20  = close.rolling(20).mean()
sma50  = close.rolling(50).mean()
sma200 = close.rolling(200).mean()

bb_mid   = close.rolling(20).mean()
bb_std   = close.rolling(20).std()
bb_upper = bb_mid + 2 * bb_std
bb_lower = bb_mid - 2 * bb_std

_delta = close.diff()
_gain  = _delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
_loss  = (-_delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
rsi    = 100 - 100 / (1 + _gain / _loss.replace(0, np.nan))

# 出来高の色（上昇日=緑、下落日=赤）
vol_colors = ['#26a69a' if c >= o else '#ef5350'
              for c, o in zip(close.values, open_.values)]

# ----- プロット -----
fig = plt.figure(figsize=(14, 10))
gs  = GridSpec(3, 1, figure=fig, height_ratios=[3, 1, 1], hspace=0.08)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax3 = fig.add_subplot(gs[2], sharex=ax1)

fig.suptitle(f"{ticker_symbol}  株価チャート  ({start_date} ～ {end_date})",
             fontsize=14, y=1.01)

# --- Panel 1: 終値 + SMA + ボリンジャーバンド ---
ax1.fill_between(dates, bb_lower, bb_upper, alpha=0.10, color='gray', label='BB ±2σ')
ax1.plot(dates, bb_upper, color='gray', linewidth=0.7, linestyle=':')
ax1.plot(dates, bb_lower, color='gray', linewidth=0.7, linestyle=':')
ax1.plot(dates, close,  label='終値',   color='#1565C0', linewidth=1.4, zorder=3)
if not sma20.isna().all():
    ax1.plot(dates, sma20,  label='SMA20',  color='#FF8F00', linewidth=1.1, linestyle='--')
if not sma50.isna().all():
    ax1.plot(dates, sma50,  label='SMA50',  color='#2E7D32', linewidth=1.1, linestyle='--')
if not sma200.isna().all():
    ax1.plot(dates, sma200, label='SMA200', color='#C62828', linewidth=1.1, linestyle='--')
ax1.set_ylabel('価格', fontsize=11)
ax1.legend(loc='upper left', fontsize=8, framealpha=0.7, ncol=3)
ax1.grid(True, alpha=0.25)
plt.setp(ax1.get_xticklabels(), visible=False)

# --- Panel 2: 出来高 ---
ax2.bar(dates, volume, color=vol_colors, alpha=0.80, width=0.8)
vol_ma5 = volume.rolling(5).mean()
ax2.plot(dates, vol_ma5, color='navy', linewidth=1.0, label='出来高 MA5')
ax2.set_ylabel('出来高', fontsize=11)
ax2.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{x/1e8:.1f}億' if x >= 1e8
                      else (f'{x/1e6:.0f}M' if x >= 1e6 else f'{x/1e3:.0f}K')))
ax2.legend(loc='upper left', fontsize=8, framealpha=0.7)
ax2.grid(True, alpha=0.25)
plt.setp(ax2.get_xticklabels(), visible=False)

# --- Panel 3: RSI ---
ax3.plot(dates, rsi, color='#6A1B9A', linewidth=1.3, label='RSI(14)')
ax3.axhline(70, color='#ef5350', linestyle='--', linewidth=1.0, alpha=0.9)
ax3.axhline(50, color='gray',    linestyle=':',  linewidth=0.8, alpha=0.7)
ax3.axhline(30, color='#26a69a', linestyle='--', linewidth=1.0, alpha=0.9)
ax3.fill_between(dates, rsi, 70, where=(rsi >= 70), alpha=0.18, color='#ef5350', label='過熱圏')
ax3.fill_between(dates, rsi, 30, where=(rsi <= 30), alpha=0.18, color='#26a69a', label='売られ過ぎ圏')
ax3.text(dates[-1], 72, '過熱(70)', color='#ef5350', fontsize=8, ha='right')
ax3.text(dates[-1], 23, '売られ過ぎ(30)', color='#26a69a', fontsize=8, ha='right')
ax3.set_ylabel('RSI(14)', fontsize=11)
ax3.set_ylim(0, 100)
ax3.grid(True, alpha=0.25)

ax3.xaxis.set_major_locator(mdates.MonthLocator())
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y/%m'))
fig.autofmt_xdate(rotation=40)

plt.tight_layout()
chart_file = f"{ticker_symbol}_chart.png"
plt.savefig(chart_file, dpi=130, bbox_inches='tight')
plt.show()
print(f"✓ チャートを保存しました: {chart_file}")


## 5. CSVファイルとしてダウンロード

In [ ]:
# ファイル名を生成
filename = f"{ticker_symbol}_{start_date}_to_{end_date}.csv"

# データの再確認（念のため）
if stock_data.empty:
    raise ValueError("❌ データが空です。このセルを実行する前にデータを取得してください。")

# CSVファイルとして保存（元のカラム名）
try:
    stock_data.to_csv(filename, encoding='utf-8-sig')  # utf-8-sigでExcelでも文字化けしない
    print(f"✓ CSVファイルを作成しました: {filename}")
    print(f"  - 行数: {len(stock_data)}")
    print(f"  - カラム数: {len(stock_data.columns)}")
except Exception as e:
    raise RuntimeError(f"❌ CSVファイルの作成に失敗しました: {str(e)}")

# ファイルをダウンロード
try:
    print("\nダウンロードを開始します...")
    files.download(filename)
    print("✓ ダウンロード完了！")
except Exception as e:
    print(f"⚠️ ダウンロードに失敗しました: {str(e)}")
    print(f"ファイル「{filename}」は作成されていますが、ダウンロードできませんでした。")

## 6. (オプション) 日本語カラム名でもダウンロード

In [ ]:
# 日本語カラム名バージョンもダウンロードしたい場合

# stock_data_jpが存在するか確認
try:
    if stock_data_jp.empty:
        raise ValueError("データが空です")
except NameError:
    raise NameError("❌ 「stock_data_jp」が定義されていません。先にセル8を実行してください。")

# 日本語化フラグの確認
try:
    is_japanese = columns_converted_to_japanese
except NameError:
    # フラグが未定義の場合、カラム名から判定
    is_japanese = '始値' in [str(col) for col in stock_data_jp.columns]

# ファイル名を動的に決定
if is_japanese:
    filename_jp = f"{ticker_symbol}_{start_date}_to_{end_date}_日本語.csv"
else:
    filename_jp = f"{ticker_symbol}_{start_date}_to_{end_date}_copy.csv"
    print("⚠️ カラムが日本語化されていないため、ファイル名を「_copy.csv」としました")

# CSVファイルとして保存
try:
    stock_data_jp.to_csv(filename_jp, encoding='utf-8-sig')
    if is_japanese:
        print(f"✓ 日本語版CSVファイルを作成しました: {filename_jp}")
    else:
        print(f"✓ CSVファイルを作成しました: {filename_jp}")
    print(f"  - 行数: {len(stock_data_jp)}")
    print(f"  - カラム数: {len(stock_data_jp.columns)}")
    print(f"  - カラム名: {', '.join(map(str, stock_data_jp.columns))}")
except Exception as e:
    raise RuntimeError(f"❌ CSVファイルの作成に失敗しました: {str(e)}")

# ファイルをダウンロード
try:
    files.download(filename_jp)
    print("✓ ダウンロード完了！")
except Exception as e:
    print(f"⚠️ ダウンロードに失敗しました: {str(e)}")
    print(f"ファイル「{filename_jp}」は作成されていますが、ダウンロードできませんでした。")

---
## 使い方のヒント

### 主要な日本株の銘柄コード例:
- トヨタ自動車: `7203.T`
- ソニーグループ: `6758.T`
- ソフトバンクグループ: `9984.T`
- キーエンス: `6861.T`
- 三菱UFJフィナンシャル・グループ: `8306.T`

### 主要な米国株の銘柄コード例:
- Apple: `AAPL`
- Microsoft: `MSFT`
- Google (Alphabet): `GOOGL`
- Amazon: `AMZN`
- Tesla: `TSLA`

### データ間隔の設定:
- `1d`: 日次データ
- `1wk`: 週次データ
- `1mo`: 月次データ
- `1h`: 時間足（直近60日間のみ）

### カラムの説明:
- **Open (始値)**: その期間の最初の取引価格
- **High (高値)**: その期間の最高価格
- **Low (安値)**: その期間の最低価格
- **Close (終値)**: その期間の最後の取引価格
- **Adj Close (調整後終値)**: 株式分割や配当を考慮した終値
- **Volume (出来高)**: その期間の取引量

---
## 7. 上昇予想銘柄のレコメンド機能

過去の株価推移（テクニカル指標）から、今後上昇が期待できる銘柄をスコアリングしてレコメンドします。

### 評価に使用するテクニカル指標
- **移動平均線 (SMA)**: 短期(20日)・中期(50日)・長期(200日)の関係から上昇トレンドを判定
- **ゴールデンクロス**: 短期SMAが中期SMAを上抜けると上昇シグナル
- **RSI (14日)**: 売られすぎ(30未満)からの反発、適度な水準(40〜60)を加点
- **モメンタム**: 直近20日・60日のリターン
- **ボリンジャーバンド**: バンド内の相対位置から過熱感を判定
- **出来高トレンド**: 直近の出来高増加で買い意欲を確認

### ⚠️ 免責事項
本機能は過去データに基づくテクニカル分析の自動化であり、将来の株価上昇を保証するものではありません。
投資判断は自己責任で行ってください。

In [ ]:
# ===== レコメンド設定 =====

# --- 対象インデックスの選択 (True = 構成銘柄を自動取得して対象に含める) ---
use_nikkei225    = True   # 日経225      （約225銘柄）
use_topix_core30 = True   # TOPIX Core30  （約30銘柄）
use_sp500        = True   # S&P 500       （約503銘柄）
use_nasdaq100    = True   # NASDAQ-100    （約100銘柄）

# --- 個別追加銘柄（インデックス外で追加したい銘柄をここに記入）---
custom_tickers = {
    # 例: "9983.T": "ファーストリテイリング",
    #     "AAPL":   "Apple",
}

# --- Stage 2（深層テクニカル分析）パラメータ ---
lookback_days   = 250   # 過去取得日数（200日SMA算出のため250推奨）
top_n           = 5     # 最終レコメンド表示件数
min_data_points = 60    # データ不足と判定する最低営業日数

# --- Stage 1（事前スクリーニング）パラメータ ---
prescreening_top_n          = 200   # Stage 1 後に深層分析へ進む最大件数
prescreening_min_20d_return = -10.0 # Stage 1 フィルタ: 20日リターン下限 (%)

print('レコメンド設定:')
print(f"  日経225       : {'有効' if use_nikkei225    else '無効'}")
print(f"  TOPIX Core30  : {'有効' if use_topix_core30 else '無効'}")
print(f"  S&P 500       : {'有効' if use_sp500        else '無効'}")
print(f"  NASDAQ-100    : {'有効' if use_nasdaq100    else '無効'}")
print(f'  カスタム銘柄  : {len(custom_tickers)} 件')
print(f'  事前スクリーニング上位 : {prescreening_top_n} 件 → 深層分析へ')
print(f'  20日リターン下限       : {prescreening_min_20d_return}%')
print(f'  最終レコメンド件数     : {top_n} 件')


### 7-2. 指数構成銘柄の自動取得

Wikipediaから各インデックスの構成銘柄を自動取得し、`candidate_tickers` を組み立てます。  
取得に失敗した場合は主要銘柄のフォールバックリストを使用します。

In [ ]:
import re as _re
from io import StringIO as _StringIO

# ============================================================
# 指数構成銘柄の取得関数（Wikipedia + フォールバック）
# ============================================================

_WIKI_HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                   'AppleWebKit/537.36 (KHTML, like Gecko) '
                   'Chrome/120.0 Safari/537.36'),
}


def _flatten_columns(df):
    'MultiIndex（2段ヘッダー）列を _ 区切りの1段文字列に平坦化する'
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy()
        df.columns = [
            '_'.join(str(v) for v in col
                     if str(v).lower() not in ('nan', ''))
            for col in df.columns
        ]
    else:
        df = df.copy()
        df.columns = [str(c) for c in df.columns]
    return df


def _get_html_tables(url: str) -> list:
    'URLからHTMLテーブルを取得（StringIOでFutureWarning回避＋列名を平坦化）'
    try:
        import requests
        resp = requests.get(url, headers=_WIKI_HEADERS, timeout=20)
        resp.raise_for_status()
        tables = pd.read_html(_StringIO(resp.text))
    except ImportError:
        tables = pd.read_html(url)
    return [_flatten_columns(t) for t in tables]


def fetch_sp500() -> dict:
    'Wikipedia から S&P 500 構成銘柄を取得'
    FALLBACK = {
        'AAPL': 'Apple', 'MSFT': 'Microsoft', 'GOOGL': 'Alphabet',
        'AMZN': 'Amazon', 'NVDA': 'NVIDIA', 'META': 'Meta Platforms',
        'TSLA': 'Tesla', 'BRK-B': 'Berkshire Hathaway',
        'JPM': 'JPMorgan Chase', 'UNH': 'UnitedHealth',
    }
    try:
        url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
        tables = _get_html_tables(url)
        df = tables[0]
        sym_col  = next(c for c in df.columns if 'Symbol'   in str(c))
        name_col = next(c for c in df.columns if 'Security' in str(c))
        result = {}
        for _, row in df.iterrows():
            sym  = str(row[sym_col]).strip()
            name = str(row[name_col]).strip()
            if sym and sym != 'nan':
                sym = _re.sub(r'(?<=[A-Z])\.(?=[A-Z])', '-', sym)  # BRK.B -> BRK-B
                result[sym] = name
        print(f'  S&P 500      : {len(result):4d} 件取得 (Wikipedia)')
        return result
    except Exception as e:
        print(f'  [警告] S&P 500 取得失敗 ({type(e).__name__}) → フォールバック使用')
        return FALLBACK


def fetch_nasdaq100() -> dict:
    'Wikipedia から NASDAQ-100 構成銘柄を取得'
    FALLBACK = {
        'AAPL': 'Apple', 'MSFT': 'Microsoft', 'NVDA': 'NVIDIA',
        'AMZN': 'Amazon', 'META': 'Meta Platforms', 'GOOGL': 'Alphabet',
        'TSLA': 'Tesla', 'AVGO': 'Broadcom', 'COST': 'Costco', 'ASML': 'ASML',
    }
    try:
        url = 'https://en.wikipedia.org/wiki/Nasdaq-100'
        tables = _get_html_tables(url)
        df = None
        for t in tables:
            cols_lower = [str(c).lower() for c in t.columns]
            has_sym  = any('ticker' in c or 'symbol' in c for c in cols_lower)
            has_name = any('company' in c or 'name' in c for c in cols_lower)
            if has_sym and has_name and len(t) > 20:
                df = t
                break
        if df is None:
            raise ValueError('構成銘柄テーブルが見つかりません')
        sym_col  = next(c for c in df.columns
                        if 'ticker' in str(c).lower() or 'symbol' in str(c).lower())
        name_col = next(c for c in df.columns
                        if 'company' in str(c).lower() or 'name' in str(c).lower())
        result = {
            str(row[sym_col]).strip(): str(row[name_col]).strip()
            for _, row in df.iterrows()
            if str(row[sym_col]).strip() not in ('', 'nan')
        }
        print(f'  NASDAQ-100   : {len(result):4d} 件取得 (Wikipedia)')
        return result
    except Exception as e:
        print(f'  [警告] NASDAQ-100 取得失敗 ({type(e).__name__}) → フォールバック使用')
        return FALLBACK


def _jp_code_to_ticker(raw) -> 'str | None':
    '日本株コード（数値またはストリング）を 1234.T 形式に変換'
    s = str(raw).strip().replace('.T', '').replace('.t', '')
    if not s or s == 'nan':
        return None
    try:
        code = str(int(float(s)))
        return f'{code}.T' if code.isdigit() else None
    except (ValueError, OverflowError):
        return None


def _scrape_jp_index(urls, min_rows, label):
    '日本語/英語Wikipediaから コード列を持つテーブルを探して dict を返す（失敗時 None）'
    JP_CODE_KEYS = ['コード', 'code', 'ticker', 'symbol', '証券コード']
    JP_NAME_KEYS = ['銘柄', '企業', '会社', 'name', 'company']
    for url in urls:
        try:
            tables = _get_html_tables(url)
        except Exception:
            continue
        for t in tables:
            cols = [str(c) for c in t.columns]  # 取得時に平坦化済み
            cols_l = [c.lower() for c in cols]
            code_col = next((cols[i] for i, c in enumerate(cols_l)
                             if any(k.lower() in c for k in JP_CODE_KEYS)), None)
            if code_col is None or len(t) < min_rows:
                continue
            name_col = next((cols[i] for i, c in enumerate(cols_l)
                             if any(k.lower() in c for k in JP_NAME_KEYS)), None)
            if name_col is None:
                name_col = cols[1] if len(cols) > 1 else code_col
            result = {}
            for _, row in t.iterrows():
                ticker = _jp_code_to_ticker(row[code_col])
                if ticker:
                    result[ticker] = str(row[name_col]).strip()
            if len(result) >= min_rows:
                return result
    return None


NIKKEI_FALLBACK = {
        '7203.T': 'トヨタ自動車',
        '7267.T': '本田技研工業',
        '7201.T': '日産自動車',
        '7269.T': 'スズキ',
        '7270.T': 'SUBARU',
        '7202.T': 'いすゞ自動車',
        '7205.T': '日野自動車',
        '7211.T': '三菱自動車',
        '7261.T': 'マツダ',
        '7272.T': 'ヤマハ発動機',
        '6902.T': 'デンソー',
        '5108.T': 'ブリヂストン',
        '5101.T': '横浜ゴム',
        '7259.T': 'アイシン',
        '6758.T': 'ソニーグループ',
        '6861.T': 'キーエンス',
        '8035.T': '東京エレクトロン',
        '6501.T': '日立製作所',
        '6503.T': '三菱電機',
        '6502.T': '東芝',
        '6752.T': 'パナソニック ホールディングス',
        '6701.T': 'NEC',
        '6702.T': '富士通',
        '6753.T': 'シャープ',
        '6954.T': 'ファナック',
        '6506.T': '安川電機',
        '6504.T': '富士電機',
        '6645.T': 'オムロン',
        '6971.T': '京セラ',
        '6762.T': 'TDK',
        '6981.T': '村田製作所',
        '6857.T': 'アドバンテスト',
        '6920.T': 'レーザーテック',
        '6146.T': 'ディスコ',
        '6723.T': 'ルネサスエレクトロニクス',
        '6724.T': 'セイコーエプソン',
        '6770.T': 'アルプスアルパイン',
        '6841.T': '横河電機',
        '6976.T': '太陽誘電',
        '6988.T': '日東電工',
        '7751.T': 'キヤノン',
        '7752.T': 'リコー',
        '7733.T': 'オリンパス',
        '7741.T': 'HOYA',
        '7731.T': 'ニコン',
        '7762.T': 'シチズン時計',
        '6479.T': 'ミネベアミツミ',
        '6594.T': 'ニデック',
        '6586.T': 'マキタ',
        '6592.T': 'マブチモーター',
        '6448.T': 'ブラザー工業',
        '6674.T': 'ジーエス・ユアサ',
        '6952.T': 'カシオ計算機',
        '6301.T': 'コマツ',
        '6326.T': 'クボタ',
        '6305.T': '日立建機',
        '6302.T': '住友重機械工業',
        '6361.T': '荏原製作所',
        '6367.T': 'ダイキン工業',
        '6471.T': '日本精工',
        '6472.T': 'NTN',
        '6473.T': 'ジェイテクト',
        '6103.T': 'オークマ',
        '6113.T': 'アマダ',
        '6273.T': 'SMC',
        '7011.T': '三菱重工業',
        '7012.T': '川崎重工業',
        '7013.T': 'IHI',
        '5631.T': '日本製鋼所',
        '7003.T': '三井E&S',
        '4063.T': '信越化学工業',
        '4005.T': '住友化学',
        '4188.T': '三菱ケミカルグループ',
        '3407.T': '旭化成',
        '4183.T': '三井化学',
        '4042.T': '東ソー',
        '4021.T': '日産化学',
        '4208.T': 'UBE',
        '3402.T': '東レ',
        '3401.T': '帝人',
        '3101.T': '東洋紡',
        '3105.T': '日清紡ホールディングス',
        '4452.T': '花王',
        '4612.T': '日本ペイントホールディングス',
        '4631.T': 'DIC',
        '4901.T': '富士フイルムホールディングス',
        '4902.T': 'コニカミノルタ',
        '4911.T': '資生堂',
        '3861.T': '王子ホールディングス',
        '3863.T': '日本製紙',
        '5201.T': 'AGC',
        '5232.T': '住友大阪セメント',
        '5233.T': '太平洋セメント',
        '5301.T': '東海カーボン',
        '5332.T': 'TOTO',
        '5333.T': '日本碍子',
        '4502.T': '武田薬品工業',
        '4503.T': 'アステラス製薬',
        '4506.T': '住友ファーマ',
        '4507.T': '塩野義製薬',
        '4519.T': '中外製薬',
        '4523.T': 'エーザイ',
        '4568.T': '第一三共',
        '4151.T': '協和キリン',
        '4543.T': 'テルモ',
        '5401.T': '日本製鉄',
        '5406.T': '神戸製鋼所',
        '5411.T': 'JFEホールディングス',
        '5703.T': '日本軽金属ホールディングス',
        '5706.T': '三井金属鉱業',
        '5711.T': '三菱マテリアル',
        '5713.T': '住友金属鉱山',
        '5714.T': 'DOWAホールディングス',
        '5801.T': '古河電気工業',
        '5802.T': '住友電気工業',
        '5803.T': 'フジクラ',
        '2502.T': 'アサヒグループホールディングス',
        '2503.T': 'キリンホールディングス',
        '2501.T': 'サッポロホールディングス',
        '2531.T': '宝ホールディングス',
        '2801.T': 'キッコーマン',
        '2802.T': '味の素',
        '2914.T': '日本たばこ産業',
        '2002.T': '日清製粉グループ本社',
        '2269.T': '明治ホールディングス',
        '2282.T': '日本ハム',
        '2871.T': 'ニチレイ',
        '2897.T': '日清食品ホールディングス',
        '8058.T': '三菱商事',
        '8031.T': '三井物産',
        '8001.T': '伊藤忠商事',
        '8002.T': '丸紅',
        '8053.T': '住友商事',
        '8015.T': '豊田通商',
        '9983.T': 'ファーストリテイリング',
        '8267.T': 'イオン',
        '3382.T': 'セブン&アイ・ホールディングス',
        '3086.T': 'Jフロント リテイリング',
        '3099.T': '三越伊勢丹ホールディングス',
        '8233.T': '高島屋',
        '4732.T': 'ユー・エス・エス',
        '8306.T': '三菱UFJフィナンシャル・グループ',
        '8316.T': '三井住友フィナンシャルグループ',
        '8411.T': 'みずほフィナンシャルグループ',
        '8308.T': 'りそなホールディングス',
        '8309.T': '三井住友トラスト・ホールディングス',
        '8304.T': 'あおぞら銀行',
        '8331.T': '千葉銀行',
        '8354.T': 'ふくおかフィナンシャルグループ',
        '7186.T': 'コンコルディア・フィナンシャルグループ',
        '8591.T': 'オリックス',
        '8601.T': '大和証券グループ本社',
        '8604.T': '野村ホールディングス',
        '8628.T': '松井証券',
        '8697.T': '日本取引所グループ',
        '8253.T': 'クレディセゾン',
        '8252.T': '丸井グループ',
        '8766.T': '東京海上ホールディングス',
        '8725.T': 'MS&ADインシュアランスグループ',
        '8630.T': 'SOMPOホールディングス',
        '8750.T': '第一生命ホールディングス',
        '8795.T': 'T&Dホールディングス',
        '8801.T': '三井不動産',
        '8802.T': '三菱地所',
        '8830.T': '住友不動産',
        '3289.T': '東急不動産ホールディングス',
        '1925.T': '大和ハウス工業',
        '1928.T': '積水ハウス',
        '1801.T': '大成建設',
        '1802.T': '大林組',
        '1803.T': '清水建設',
        '1808.T': '長谷工コーポレーション',
        '1812.T': '鹿島建設',
        '1963.T': '日揮ホールディングス',
        '9020.T': '東日本旅客鉄道',
        '9021.T': '西日本旅客鉄道',
        '9022.T': '東海旅客鉄道',
        '9001.T': '東武鉄道',
        '9005.T': '東急',
        '9007.T': '小田急電鉄',
        '9008.T': '京王電鉄',
        '9009.T': '京成電鉄',
        '9101.T': '日本郵船',
        '9104.T': '商船三井',
        '9107.T': '川崎汽船',
        '9064.T': 'ヤマトホールディングス',
        '9147.T': 'NIPPON EXPRESSホールディングス',
        '9201.T': '日本航空',
        '9202.T': 'ANAホールディングス',
        '9301.T': '三菱倉庫',
        '9432.T': 'NTT',
        '9433.T': 'KDDI',
        '9434.T': 'ソフトバンク',
        '9984.T': 'ソフトバンクグループ',
        '9613.T': 'NTTデータグループ',
        '6098.T': 'リクルートホールディングス',
        '4307.T': '野村総合研究所',
        '4324.T': '電通グループ',
        '4689.T': 'LINEヤフー',
        '4704.T': 'トレンドマイクロ',
        '4751.T': 'サイバーエージェント',
        '4385.T': 'メルカリ',
        '9735.T': 'セコム',
        '2412.T': 'ベネッセホールディングス',
        '9501.T': '東京電力ホールディングス',
        '9502.T': '中部電力',
        '9503.T': '関西電力',
        '9531.T': '東京ガス',
        '9532.T': '大阪ガス',
        '1605.T': 'INPEX',
        '5020.T': 'ENEOSホールディングス',
        '5019.T': '出光興産',
        '6178.T': '日本郵政',
        '7974.T': '任天堂',
        '7832.T': 'バンダイナムコホールディングス',
        '9766.T': 'コナミグループ',
        '9602.T': '東宝',
        '4661.T': 'オリエンタルランド',
        '7912.T': '大日本印刷',
        '7911.T': 'TOPPANホールディングス',
        '7951.T': 'ヤマハ',
    }

TOPIX_CORE30_FALLBACK = {
        '7203.T': 'トヨタ自動車',
        '6758.T': 'ソニーグループ',
        '6861.T': 'キーエンス',
        '8306.T': '三菱UFJフィナンシャル・グループ',
        '9983.T': 'ファーストリテイリング',
        '9984.T': 'ソフトバンクグループ',
        '6098.T': 'リクルートホールディングス',
        '8035.T': '東京エレクトロン',
        '6501.T': '日立製作所',
        '9432.T': 'NTT',
        '9433.T': 'KDDI',
        '4063.T': '信越化学工業',
        '6954.T': 'ファナック',
        '4519.T': '中外製薬',
        '4568.T': '第一三共',
        '8058.T': '三菱商事',
        '8031.T': '三井物産',
        '8316.T': '三井住友フィナンシャルグループ',
        '6367.T': 'ダイキン工業',
        '6902.T': 'デンソー',
        '7267.T': '本田技研工業',
        '6981.T': '村田製作所',
        '2914.T': '日本たばこ産業',
        '4502.T': '武田薬品工業',
        '8411.T': 'みずほフィナンシャルグループ',
        '8766.T': '東京海上ホールディングス',
        '6594.T': 'ニデック',
        '7974.T': '任天堂',
        '4661.T': 'オリエンタルランド',
        '6146.T': 'ディスコ',
    }


def _fetch_nikkei_official() -> 'dict | None':
    '日経公式の構成銘柄CSVから取得する（Shift-JIS等を自動判定、列順に依存しない）'
    url = ('https://indexes.nikkei.co.jp/nkave/archives/file/'
           'nikkei_stock_average_components_jp.csv')
    try:
        import requests, csv as _csv
        resp = requests.get(url, headers=_WIKI_HEADERS, timeout=20)
        resp.raise_for_status()
        raw = resp.content
    except Exception:
        return None
    text = None
    for enc in ('shift_jis', 'cp932', 'utf-8-sig', 'utf-8'):
        try:
            text = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    if text is None:
        return None
    result = {}
    for row in _csv.reader(_StringIO(text)):
        # 4桁数字のセルを証券コードとみなす（列順に非依存）
        code_idx = next((i for i, c in enumerate(row)
                         if c.strip().isdigit() and len(c.strip()) == 4), None)
        if code_idx is None:
            continue
        code = row[code_idx].strip()
        name = next((c.strip() for j, c in enumerate(row)
                     if j != code_idx and c.strip() and not c.strip().isdigit()),
                    code)
        result[f'{code}.T'] = name
    return result if len(result) >= 100 else None


def fetch_nikkei225() -> dict:
    '日経225 構成銘柄を取得（日経公式CSV → Wikipedia → 静的リスト）'
    # 1) 日経公式の構成銘柄CSV（最も正確）
    result = _fetch_nikkei_official()
    if result:
        print(f'  日経225      : {len(result):4d} 件取得 (日経公式CSV)')
        return result
    # 2) Wikipedia
    urls = [
        'https://ja.wikipedia.org/wiki/日経平均株価',
        'https://ja.wikipedia.org/wiki/日経225銘柄一覧',
        'https://en.wikipedia.org/wiki/Nikkei_225',
    ]
    result = _scrape_jp_index(urls, min_rows=100, label='日経225')
    if result:
        print(f'  日経225      : {len(result):4d} 件取得 (Wikipedia)')
        return result
    # 3) 静的フォールバック
    print(f'  [情報] 日経225 はオンライン取得不可 → 主要 {len(NIKKEI_FALLBACK)} 銘柄の静的リストを使用')
    return dict(NIKKEI_FALLBACK)


def fetch_topix_core30() -> dict:
    'TOPIX Core30 構成銘柄を取得（日本語版Wikipedia優先 → 静的リスト）'
    urls = [
        'https://ja.wikipedia.org/wiki/TOPIX_Core30',
        'https://en.wikipedia.org/wiki/TOPIX_Core_30',
        'https://en.wikipedia.org/wiki/TOPIX_Core30',
    ]
    result = _scrape_jp_index(urls, min_rows=25, label='TOPIX Core30')
    if result and len(result) <= 40:  # Core30 は約30銘柄。多すぎる場合は誤取得
        print(f'  TOPIX Core30 : {len(result):4d} 件取得 (Wikipedia)')
        return result
    if result:
        print(f'  [情報] TOPIX Core30 Wikipedia取得結果({len(result)}件)が多すぎるため除外')
    print(f'  [情報] TOPIX Core30 はWikipedia取得不可 → 構成 {len(TOPIX_CORE30_FALLBACK)} 銘柄の静的リストを使用')
    return dict(TOPIX_CORE30_FALLBACK)


# ============================================================
# candidate_tickers の組み立て
# ============================================================
print('インデックス構成銘柄を取得中...\n')

candidate_tickers: dict = {}

if use_nikkei225:
    candidate_tickers.update(fetch_nikkei225())
if use_topix_core30:
    candidate_tickers.update(fetch_topix_core30())
if use_sp500:
    candidate_tickers.update(fetch_sp500())
if use_nasdaq100:
    candidate_tickers.update(fetch_nasdaq100())

# カスタム銘柄を最後にマージ（最優先）
candidate_tickers.update(custom_tickers)

print(f'\n合計候補銘柄数: {len(candidate_tickers)} 件')
print(f'（Stage 1 スクリーニング後に上位 {prescreening_top_n} 件を深層分析します）')


### 7-3. Stage 1: 事前スクリーニング（一括ダウンロード）

全候補銘柄の直近3ヶ月データを一括取得し、**20日リターン**と**出来高比**で絞り込みます。  
上位 `prescreening_top_n` 件のみを Stage 2（深層テクニカル分析）に進めます。

In [ ]:
import time

# ============================================================
# Stage 1: 一括ダウンロードによる事前スクリーニング
# ============================================================

BATCH_SIZE  = 100
all_tickers = list(candidate_tickers.keys())
batches     = [all_tickers[i:i + BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]

print('Stage 1 事前スクリーニング開始')
print(f'  対象銘柄数 : {len(all_tickers):5d} 件')
print(f'  バッチ数   : {len(batches):5d} バッチ（各最大 {BATCH_SIZE} 件）')
print(f'  取得期間   : 直近 3 ヶ月\n')

screening_records = []
stage1_start = time.time()

for batch_idx, batch in enumerate(batches):
    print(f'  [{batch_idx + 1:2d}/{len(batches)}] {len(batch):3d} 件をダウンロード中...', end=' ', flush=True)
    try:
        bulk_df = yf.download(
            batch,
            period='3mo',
            group_by='ticker',
            progress=False,
            auto_adjust=False,
        )
    except Exception as e:
        print(f'[エラー: {type(e).__name__}] — スキップ')
        continue

    if bulk_df is None or bulk_df.empty:
        print('[データなし] — スキップ')
        continue

    is_multi   = isinstance(bulk_df.columns, pd.MultiIndex)
    lv0        = set(bulk_df.columns.get_level_values(0).tolist()) if is_multi else set()
    field_first = ('Close' in lv0) if is_multi else True  # True: (field, ticker)

    batch_ok = 0
    for ticker in batch:
        try:
            if is_multi:
                if field_first:
                    if 'Close' not in bulk_df or ticker not in bulk_df['Close'].columns:
                        continue
                    close_s  = bulk_df['Close'][ticker].dropna()
                    volume_s = bulk_df['Volume'][ticker].dropna()
                else:
                    if ticker not in lv0:
                        continue
                    close_s  = bulk_df[ticker]['Close'].dropna()
                    volume_s = bulk_df[ticker]['Volume'].dropna()
            else:
                close_s  = bulk_df['Close'].dropna()
                volume_s = bulk_df['Volume'].dropna()

            close_s  = close_s.astype(float)
            volume_s = volume_s.astype(float)

            if len(close_s) < 21:
                continue

            ret_20d   = (float(close_s.iloc[-1]) / float(close_s.iloc[-21]) - 1) * 100
            vol_5d    = float(volume_s.tail(5).mean())
            vol_20d   = float(volume_s.tail(20).mean())
            vol_ratio = vol_5d / vol_20d if vol_20d > 0 else 0.0

            screening_records.append({
                'ticker'    : ticker,
                'name'      : candidate_tickers.get(ticker, ticker),
                'return_20d': ret_20d,
                'vol_ratio' : vol_ratio,
            })
            batch_ok += 1
        except Exception:
            continue

    print(f'OK ({batch_ok}/{len(batch)} 件)')

stage1_elapsed = time.time() - stage1_start

if not screening_records:
    raise RuntimeError('❌ Stage 1 でデータが取得できませんでした。ネットワーク接続を確認してください。')

screen_df = pd.DataFrame(screening_records)

filtered_df = screen_df[
    (screen_df['return_20d'] >= prescreening_min_20d_return) &
    (screen_df['vol_ratio']  >  0)
].copy()

filtered_df = filtered_df.sort_values('return_20d', ascending=False).head(prescreening_top_n)

prescreened_tickers = {
    row['ticker']: row['name']
    for _, row in filtered_df.iterrows()
}

print()
print('=' * 65)
print(f'Stage 1 完了  （{stage1_elapsed:.1f} 秒）')
print('=' * 65)
print(f'  データ取得成功 : {len(screen_df):4d} / {len(all_tickers)} 件')
print(f'  フィルタ通過   : {len(filtered_df):4d} 件  （20日リターン >= {prescreening_min_20d_return}%）')
print(f'  Stage 2 対象   : {len(prescreened_tickers):4d} 件')
print()
print('  スクリーニング上位 10 件:')
for i, (_, row) in enumerate(filtered_df.head(10).iterrows(), 1):
    print(f"    {i:2d}. {row['ticker']:12s}  {row['name'][:22]:22s}  "
          f"20日: {row['return_20d']:+6.1f}%  出来高比: {row['vol_ratio']:.2f}")


### 7-3.5 Stage 1.5: ファンダメンタルズ・クオリティフィルター

Stage 1 の通過銘柄に対して財務指標（yfinance）を取得し、
**赤字・過剰債務などの地雷銘柄を Stage 2 前に除外**します。
データが取得できない銘柄（日本株など）は中立として通過させます。

| 指標 | 除外条件（ハードフィルター） | スコア加減点 |
|---|---|---|
| **ROE** | < −5%（継続的赤字） | +10（≥15%）/ +5（≥8%） |
| **D/Eレシオ** | > 300（yfinance単位 ≒ D/E 3.0） | −5（> 150） |
| **PER** | 負値または > 150 | +5（10〜20）/ −10（負 or >150） |
| **利益率** | — | +5（≥ 10%） |
| **売上成長** | — | +5（≥ 10%） |
| **流動比率** | — | −10（< 1.0） |

> ⚠️ yfinance のファンダ情報は**日本株で欠損が多め**です。
> 取得できなかった指標はスコアに反映されません（除外されません）。

In [ ]:
# ===== Stage 1.5: ファンダメンタルズ・クオリティフィルター =====
import time as _time_fund

RUN_FUNDAMENTAL_FILTER = True   # False でスキップ
FUND_EXCLUDE_ROE_MIN   = -0.05  # ROE がこれ未満 → 除外（継続赤字）
FUND_EXCLUDE_DE_MAX    = 300    # D/Eレシオ（yfinance単位）がこれ超 → 除外（過剰債務）
FUND_CHECK_TOP         = 100    # 最大何銘柄まで確認するか（処理時間の上限制御）

def _fetch_fundamentals(ticker):
    """yfinance で主要ファンダ指標を取得。失敗時は空 dict を返す。"""
    try:
        info = yf.Ticker(ticker).info
        return {
            'per':           info.get('trailingPE'),
            'pbr':           info.get('priceToBook'),
            'roe':           info.get('returnOnEquity'),
            'de_ratio':      info.get('debtToEquity'),   # yfinance: × 100 表記
            'profit_margin': info.get('profitMargins'),
            'rev_growth':    info.get('revenueGrowth'),
            'current_ratio': info.get('currentRatio'),
        }
    except Exception:
        return {}

def _fund_quality(fd):
    """ファンダ dict → (スコア調整値, 除外フラグ, 理由リスト)"""
    score = 0
    exclude = False
    reasons = []

    roe = fd.get('roe')
    de  = fd.get('de_ratio')
    per = fd.get('per')
    pm  = fd.get('profit_margin')
    rg  = fd.get('rev_growth')
    cr  = fd.get('current_ratio')

    # ---- ハードフィルター（除外）----
    if roe is not None and roe < FUND_EXCLUDE_ROE_MIN:
        exclude = True; reasons.append(f'ROE={roe:.1%}(赤字)')
    if de is not None and de > FUND_EXCLUDE_DE_MAX:
        exclude = True; reasons.append(f'D/E={de:.0f}(過剰債務)')

    if exclude:
        return score, True, reasons

    # ---- ソフトスコア ----
    if roe is not None:
        if roe >= 0.15:    score += 10; reasons.append(f'ROE={roe:.1%}↑')
        elif roe >= 0.08:  score += 5;  reasons.append(f'ROE={roe:.1%}')
    if de is not None and de > 150:
        score -= 5; reasons.append(f'D/E={de:.0f}(高め)')
    if per is not None:
        if per < 0 or per > 150: score -= 10; reasons.append(f'PER={per:.1f}(過熱/赤字)')
        elif 10 <= per <= 20:    score += 5;  reasons.append(f'PER={per:.1f}(割安)')
    if pm is not None and pm >= 0.10:
        score += 5; reasons.append(f'利益率={pm:.1%}')
    if rg is not None and rg >= 0.10:
        score += 5; reasons.append(f'売上成長={rg:.1%}')
    if cr is not None and cr < 1.0:
        score -= 10; reasons.append(f'流動比率={cr:.2f}(低)')

    return score, False, reasons


if RUN_FUNDAMENTAL_FILTER:
    _tickers_to_check = list(prescreened_tickers.keys())[:FUND_CHECK_TOP]
    _n_total = len(_tickers_to_check)
    print(f'ファンダメンタルズフィルター: {_n_total} 銘柄を確認中（最大 {FUND_CHECK_TOP} 件）...')
    print('  ※ 日本株はデータ欠損が多く、取得不可の場合は中立として通過します\n')

    fund_scores = {}       # ticker -> (score_adj, excluded, reasons)
    _excluded_tickers = []

    for _i, _tkr in enumerate(_tickers_to_check, 1):
        _nm = prescreened_tickers[_tkr]
        _fd = _fetch_fundamentals(_tkr)
        _fs, _exc, _rsns = _fund_quality(_fd)
        fund_scores[_tkr] = (_fs, _exc, _fd)

        if _exc:
            _excluded_tickers.append(_tkr)
            _reason_str = ' / '.join(_rsns)
            print(f'  ❌ 除外 [{_i:3d}] {_tkr:<12} {_nm[:16]:<16}  {_reason_str}')
        elif _fs != 0:
            _reason_str = ' / '.join(_rsns) if _rsns else '—'
            print(f'  {"+" if _fs > 0 else ""}{_fs:+3d}pt [{_i:3d}] {_tkr:<12} {_nm[:16]:<16}  {_reason_str}')

        # yfinance への過負荷を避けるため小休止
        if _i % 10 == 0:
            _time_fund.sleep(0.5)

    # prescreened_tickers から除外銘柄を取り除く
    _before = len(prescreened_tickers)
    prescreened_tickers = {k: v for k, v in prescreened_tickers.items()
                           if k not in _excluded_tickers}
    print(f'\n  除外: {len(_excluded_tickers)} 銘柄  → '
          f'Stage 2 対象: {len(prescreened_tickers)} 銘柄 '
          f'（{_before} → {len(prescreened_tickers)}）')
    if _excluded_tickers:
        print(f'  除外銘柄: {", ".join(_excluded_tickers)}')
    # FUND_CHECK_TOP に満たなかった残りは中立として fund_scores に登録
    for _tkr in prescreened_tickers:
        if _tkr not in fund_scores:
            fund_scores[_tkr] = (0, False, [])
else:
    fund_scores = {}
    print('ファンダメンタルズフィルターはスキップされました（RUN_FUNDAMENTAL_FILTER = False）')

In [ ]:
# ===== テクニカル指標の計算とスコアリング =====
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta


def compute_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    """RSI（相対力指数）を計算"""
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def analyze_ticker(ticker: str, name: str, period_days: int) -> dict | None:
    """1銘柄分のテクニカル指標とスコアを算出する。データ不足ならNoneを返す。"""
    end = datetime.today()
    start = end - timedelta(days=period_days + 30)  # 余裕を持って取得

    try:
        df = yf.download(ticker, start=start.strftime("%Y-%m-%d"),
                         end=end.strftime("%Y-%m-%d"), progress=False, auto_adjust=False)
    except Exception as e:
        print(f"  [スキップ] {ticker}: 取得エラー ({e})")
        return None

    if df.empty or len(df) < min_data_points:
        print(f"  [スキップ] {ticker}: データ不足 ({len(df) if not df.empty else 0} 日)")
        return None

    # MultiIndexカラム対応（yfinanceのバージョン差を吸収）
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    close = df["Close"].astype(float)
    volume = df["Volume"].astype(float)

    # 各種指標
    sma20 = close.rolling(20).mean()
    sma50 = close.rolling(50).mean()
    sma200 = close.rolling(min(200, len(close))).mean()
    rsi14 = compute_rsi(close, 14)
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    bb_upper = bb_mid + 2 * bb_std
    bb_lower = bb_mid - 2 * bb_std

    last_close = float(close.iloc[-1])
    last_sma20 = float(sma20.iloc[-1]) if not np.isnan(sma20.iloc[-1]) else np.nan
    last_sma50 = float(sma50.iloc[-1]) if not np.isnan(sma50.iloc[-1]) else np.nan
    last_sma200 = float(sma200.iloc[-1]) if not np.isnan(sma200.iloc[-1]) else np.nan
    last_rsi = float(rsi14.iloc[-1]) if not np.isnan(rsi14.iloc[-1]) else np.nan

    # モメンタム（リターン）
    ret_20d = (last_close / float(close.iloc[-21]) - 1) * 100 if len(close) > 21 else np.nan
    ret_60d = (last_close / float(close.iloc[-61]) - 1) * 100 if len(close) > 61 else np.nan

    # 出来高トレンド: 直近5日平均 vs 過去20日平均
    vol_recent = float(volume.tail(5).mean())
    vol_base = float(volume.tail(20).mean())
    vol_ratio = vol_recent / vol_base if vol_base > 0 else 1.0

    # ボリンジャー位置（0=下限, 1=上限）
    bb_pos = np.nan
    if not np.isnan(bb_upper.iloc[-1]) and not np.isnan(bb_lower.iloc[-1]):
        rng = float(bb_upper.iloc[-1]) - float(bb_lower.iloc[-1])
        if rng > 0:
            bb_pos = (last_close - float(bb_lower.iloc[-1])) / rng

    # ===== スコアリング（合計 100点満点）=====
    score = 0.0
    reasons = []

    # 1. トレンド: 終値 > SMA20 > SMA50 > SMA200 (各10点, 最大30点)
    if not np.isnan(last_sma20) and last_close > last_sma20:
        score += 10
        reasons.append("終値>SMA20")
    if not np.isnan(last_sma50) and not np.isnan(last_sma20) and last_sma20 > last_sma50:
        score += 10
        reasons.append("SMA20>SMA50")
    if not np.isnan(last_sma200) and not np.isnan(last_sma50) and last_sma50 > last_sma200:
        score += 10
        reasons.append("SMA50>SMA200")

    # 2. ゴールデンクロス（直近10営業日以内）: 15点
    if len(sma20.dropna()) >= 11 and len(sma50.dropna()) >= 11:
        recent20 = sma20.tail(11).values
        recent50 = sma50.tail(11).values
        crossed = False
        for i in range(1, len(recent20)):
            if recent20[i - 1] <= recent50[i - 1] and recent20[i] > recent50[i]:
                crossed = True
                break
        if crossed:
            score += 15
            reasons.append("直近ゴールデンクロス")

    # 3. RSI: 売られすぎ反発(30〜45)で+15点、適度(45〜65)で+10点、過熱(>75)は減点
    if not np.isnan(last_rsi):
        if 30 <= last_rsi < 45:
            score += 15
            reasons.append(f"RSI反発圏({last_rsi:.0f})")
        elif 45 <= last_rsi <= 65:
            score += 10
            reasons.append(f"RSI健全({last_rsi:.0f})")
        elif last_rsi > 75:
            score -= 10
            reasons.append(f"RSI過熱({last_rsi:.0f})")

    # 4. モメンタム: 20日リターン(最大15点)、60日リターン(最大10点)
    if not np.isnan(ret_20d):
        score += max(min(ret_20d, 15), -10) * (15 / 15)  # 簡易マッピング
        if ret_20d > 0:
            reasons.append(f"20日+{ret_20d:.1f}%")
    if not np.isnan(ret_60d):
        score += max(min(ret_60d / 2, 10), -5)
        if ret_60d > 0:
            reasons.append(f"60日+{ret_60d:.1f}%")

    # 5. 出来高トレンド: 直近5日平均が20日平均より高ければ+5、1.5倍以上で+10
    if vol_ratio >= 1.5:
        score += 10
        reasons.append(f"出来高急増(x{vol_ratio:.1f})")
    elif vol_ratio >= 1.1:
        score += 5
        reasons.append(f"出来高増(x{vol_ratio:.2f})")

    # 6. ボリンジャーバンド: 中央付近(0.3〜0.7)で安定上昇なら+5、上限超え(>1.0)は減点
    if not np.isnan(bb_pos):
        if 0.3 <= bb_pos <= 0.7:
            score += 5
        elif bb_pos > 1.0:
            score -= 5
            reasons.append("BB上抜け(過熱)")

    # ===== エントリースコア（今が買い時かを判定）=====
    # 前提: 終値 >= SMA200（マクロ上昇トレンド内であること）
    entry_s = 0
    entry_r = []
    above_sma200 = (not np.isnan(last_sma200)) and (last_close >= last_sma200)
    if above_sma200:
        # 1. RSI回復ゾーン (30〜55): 売られすぎからの回復・健全な水準
        if not np.isnan(last_rsi) and 30 <= last_rsi <= 55:
            entry_s += 30
            entry_r.append(f'RSI回復({last_rsi:.0f})')
        # 2. 価格がSMA20を直近5日で下から上抜け（反転シグナル）
        if (not np.isnan(last_sma20) and last_close > last_sma20
                and len(close) >= 6 and len(sma20.dropna()) >= 6):
            win = [(float(close.iloc[i]), float(sma20.iloc[i]))
                   for i in range(-6, -1) if not np.isnan(sma20.iloc[i])]
            if win and any(c < s for c, s in win):
                entry_s += 25
                entry_r.append('SMA20上抜け')
        # 3. ゴールデンクロス直近20日（エントリー窓を広げて検出）
        if len(sma20.dropna()) >= 21 and len(sma50.dropna()) >= 21:
            r20e = sma20.tail(21).values
            r50e = sma50.tail(21).values
            if any(r20e[i-1] <= r50e[i-1] and r20e[i] > r50e[i]
                   for i in range(1, len(r20e))):
                entry_s += 20
                entry_r.append('GC直近20日')
        # 4. 価格がBBミッド以下（まだ上昇余地あり）
        bb_mid_last = float(bb_mid.iloc[-1]) if not np.isnan(bb_mid.iloc[-1]) else np.nan
        if not np.isnan(bb_mid_last) and last_close < bb_mid_last:
            entry_s += 15
            entry_r.append('BB下半分')
        # 5. 出来高急増（直近5日 > 20日平均 × 1.2）
        if vol_ratio > 1.2:
            entry_s += 15
            entry_r.append(f'出来高急増(x{vol_ratio:.1f})')


    return {
        "ticker": ticker,
        "name": name,
        "終値": round(last_close, 2),
        "SMA20": round(last_sma20, 2) if not np.isnan(last_sma20) else None,
        "SMA50": round(last_sma50, 2) if not np.isnan(last_sma50) else None,
        "SMA200": round(last_sma200, 2) if not np.isnan(last_sma200) else None,
        "RSI14": round(last_rsi, 1) if not np.isnan(last_rsi) else None,
        "20日リターン%": round(ret_20d, 2) if not np.isnan(ret_20d) else None,
        "60日リターン%": round(ret_60d, 2) if not np.isnan(ret_60d) else None,
        "出来高比": round(vol_ratio, 2),
        "BB位置": round(bb_pos, 2) if not np.isnan(bb_pos) else None,
        "スコア": round(score, 1),
        "シグナル": " / ".join(reasons) if reasons else "-",
        "entry_score": round(entry_s, 1),
        "entry_reasons": " / ".join(entry_r) if entry_r else "-",
    }


print("テクニカル分析の関数を定義しました。")


In [ ]:
# ===== Stage 2: 深層テクニカル分析 & 上昇期待度ランキング =====
import time as _time2

stage2_start  = _time2.time()
total_stage2  = len(prescreened_tickers)
print(f'Stage 2 深層分析: {total_stage2} 銘柄 × 直近 {lookback_days} 日分を分析中...\n')

results = []
for i, (tkr, nm) in enumerate(prescreened_tickers.items(), 1):
    pct = i / total_stage2 * 100
    print(f'  [{i:3d}/{total_stage2}] {pct:4.0f}%  {tkr} ({nm})')
    r = analyze_ticker(tkr, nm, lookback_days)
    if r is not None:
        results.append(r)

stage2_elapsed = _time2.time() - stage2_start
try:
    _total_str = f' / 合計 {stage1_elapsed + stage2_elapsed:.0f} 秒'
except NameError:
    _total_str = ''
print(f'\nStage 2 完了  （{stage2_elapsed:.1f} 秒{_total_str}）\n')

if not results:
    raise RuntimeError('❌ 分析可能なデータが取得できませんでした。銘柄リストやネットワークを確認してください。')

# DataFrameに集約し、スコアで降順ソート
ranking_df = pd.DataFrame(results).sort_values("スコア", ascending=False).reset_index(drop=True)
ranking_df.index = ranking_df.index + 1  # 1始まり
ranking_df.index.name = "順位"

# ファンダメンタルズスコアを加算してランキングを再ソート
if fund_scores:
    ranking_df['ファンダスコア'] = ranking_df['ticker'].map(
        lambda t: fund_scores.get(t, (0, False, {}))[0])
    ranking_df['スコア'] = (ranking_df['スコア'] + ranking_df['ファンダスコア']).round(1)
    ranking_df = ranking_df.sort_values('スコア', ascending=False).reset_index(drop=True)
    ranking_df.index += 1
    ranking_df.index.name = "順位"
    _n_adj = (ranking_df['ファンダスコア'] != 0).sum()
    print(f'  ✓ ファンダ調整: {_n_adj} 銘柄のスコアを修正')

print("\n" + "=" * 70)
print(f"📈 上昇期待度ランキング TOP {top_n}")
print("=" * 70)

top_df = ranking_df.head(top_n)
for rank, row in top_df.iterrows():
    print(f"\n第{rank}位: {row['ticker']} ({row['name']})  スコア: {row['スコア']}")
    print(f"  終値 {row['終値']} / RSI {row['RSI14']} / 20日 {row['20日リターン%']}% / 60日 {row['60日リターン%']}%")
    print(f"  シグナル: {row['シグナル']}")

print("\n" + "=" * 70)
print("全銘柄のスコア表:")
print("=" * 70)
display_cols = ["ticker", "name", "スコア", "終値", "RSI14", "20日リターン%", "60日リターン%", "出来高比", "シグナル"]
print(ranking_df[display_cols].to_string())

# ----- 狙い目銘柄ランキング（今が買い時エントリー評価）-----
_entry_list = [r for r in results if r.get('entry_score', 0) > 0]
if _entry_list:
    entry_df = (pd.DataFrame(_entry_list)
                .sort_values('entry_score', ascending=False)
                .reset_index(drop=True))
    entry_df.index += 1
    print('\n' + '=' * 70)
    _eh = min(5, len(entry_df))
    print(f'🎯 狙い目銘柄 TOP {_eh}  （上昇の入口にいると判定された銘柄）')
    print('=' * 70)
    print('  ※ テクニカル上位とは別に「今が買い時」という観点でスクリーニング')
    print('  ※ RSI回復 / SMA20上抜け / GC直後 / BB下半分 / 出来高増 を評価\n')
    for idx, row in entry_df.head(5).iterrows():
        print(f"  #{idx}  {row['ticker']} ({row['name']})")
        print(f"      狙い目スコア: {row['entry_score']}  テクニカルスコア: {row['スコア']}")
        print(f"      エントリー理由: {row['entry_reasons']}")
else:
    entry_df = pd.DataFrame()
    print('\n🎯 狙い目銘柄: 現時点でエントリー条件を満たす銘柄がありません')

# CSV出力
recommend_filename = f"recommend_top_{datetime.today().strftime('%Y%m%d')}.csv"
try:
    ranking_df.to_csv(recommend_filename, encoding="utf-8-sig")
    print(f"\n✓ ランキングをCSVに保存しました: {recommend_filename}")
    try:
        from google.colab import files as colab_files
        colab_files.download(recommend_filename)
    except Exception:
        print("  (Colab以外の環境ではダウンロードはスキップされます)")
except Exception as e:
    print(f"⚠️ CSV保存に失敗: {e}")

print("\n⚠️ 免責事項: このスコアは過去データからのテクニカル分析であり、将来の値上がりを保証するものではありません。")
print("   投資判断はご自身の責任で行ってください。")

# ----- スコアのバーチャート -----
fig, axes2 = plt.subplots(1, 2, figsize=(16, max(4, len(ranking_df) * 0.5 + 1)))

# 左: スコアランキング（横棒グラフ）
ax_bar = axes2[0]
labels = [f"{row['ticker']} ({row['name']})" for _, row in ranking_df.iterrows()]
scores_list = ranking_df['スコア'].tolist()
bar_colors = ['#1565C0' if i < top_n else '#B0BEC5' for i in range(len(ranking_df))]
hbars = ax_bar.barh(labels, scores_list, color=bar_colors, edgecolor='white', height=0.65)
for bar, sc in zip(hbars, scores_list):
    ax_bar.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{sc:.1f}', va='center', fontsize=9)
ax_bar.set_xlabel('スコア', fontsize=11)
ax_bar.set_title('上昇期待度スコア ランキング', fontsize=12)
ax_bar.invert_yaxis()
ax_bar.set_xlim(0, max(scores_list) * 1.18 if max(scores_list) > 0 else 10)
ax_bar.axvline(50, color='orange', linestyle='--', linewidth=0.9, alpha=0.7)
ax_bar.grid(axis='x', alpha=0.3)

# 右: 上位銘柄のテクニカル指標ヒートマップ
ax_heat = axes2[1]
heat_cols = ['RSI14', '20日リターン%', '60日リターン%', '出来高比']
heat_labels = ['RSI14', '20日\nリターン%', '60日\nリターン%', '出来高比']
top_data = top_df[heat_cols].copy()
top_data.index = [row['ticker'] for _, row in top_df.iterrows()]
top_data = top_data.astype(float).fillna(0)
_norm = top_data.copy()
for col in _norm.columns:
    mn, mx = _norm[col].min(), _norm[col].max()
    _norm[col] = (_norm[col] - mn) / (mx - mn + 1e-9)
im = ax_heat.imshow(_norm.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax_heat.set_xticks(range(len(heat_labels)))
ax_heat.set_xticklabels(heat_labels, fontsize=9)
ax_heat.set_yticks(range(len(top_data.index)))
ax_heat.set_yticklabels(top_data.index, fontsize=9)
ax_heat.set_title(f'TOP {top_n} テクニカル指標ヒートマップ', fontsize=12)
for i in range(len(top_data.index)):
    for j in range(len(heat_cols)):
        val = top_data.iloc[i, j]
        nv = _norm.iloc[i, j]
        ax_heat.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=9,
                     color='black' if 0.2 < nv < 0.8 else 'white')
plt.colorbar(im, ax=ax_heat, shrink=0.8, label='相対スコア')

plt.tight_layout()
plt.savefig('recommend_chart.png', dpi=130, bbox_inches='tight')
plt.show()
print("\n✓ レコメンドチャートを保存しました: recommend_chart.png")


---
## 8. AI予測モデルによる将来株価予測（Stage 3）

Stage 2 で抽出した上位銘柄に対し、**4種類のニューラルネットワーク**で訓練し、
**最適モデルを自動選択**して将来の株価を予測します。

### 比較する4モデル
| モデル | 構造 | 特徴 |
|---|---|---|
| **LSTM-Simple**   | LSTM(32) + Dense(1) | 軽量・高速 |
| **LSTM-Stacked**  | LSTM(50) × 3 + Dropout + Dense(1) | 表現力高め |
| **GRU**           | GRU(50) × 2 + Dropout + Dense(1) | LSTM の軽量代替 |
| **Transformer**   | MultiHeadAttention + LayerNorm + GAP + Dense | 長距離依存に強い |

### 入力特徴量（多変量モード `NN_USE_FEATURES=True`）
| 特徴量 | 説明 |
|---|---|
| close_logret | 終値の対数収益率（予測ターゲット） |
| vol_logret | 出来高の対数変化 |
| volatility | 直近10日の対数収益率の標準偏差 |
| rsi_norm | RSI(14) を [-1,1] に正規化 |
| sma20_dist | 終値/SMA20 − 1（移動平均乖離） |

### 評価指標と読み方
| 指標 | 意味 | 目安 |
|---|---|---|
| **MAPE (%)** | 価格予測の誤差率 — 小さいほど精度が高い | 2〜5% が目安 |
| **方向精度 (Dir Acc)** | 翌日「上がるか下がるか」の正解率 | 50%=ランダム、55%以上で有意 |
| **ナイーブ比較** | ランダムウォーク（前日価格据え置き）との比較 | NN < ナイーブ MAPE で「有効」 |
| **予測区間** | MC Dropout による不確実性バンド | 区間幅が大きいほど不確実 |

### 🔑 有望銘柄の見方（複合シグナル）
テクニカルスコアと AI 予測の**両方が上昇を示す銘柄**が最も強いシグナルです。
最後のセルに **「複合スコアランキング」** を表示します。

- **✅NN有効**: NN の MAPE がランダムウォーク（ナイーブ）を下回る → 予測が意味を持つ
- **⚠️ナイーブ以下**: NN がナイーブより精度が低い → 予測の信頼度が低い（見送り推奨）

> **技術的補足**: 本実装では**対数収益率**を学習対象とし、
> MC Dropout（推論時 Dropout 有効）で複数の予測パスを生成して**不確実性バンド**を算出します。
> 将来の出来高は未知なため、外生特徴（vol_logret, rsi_norm）は最終観測値を据え置きます。

### ⚠️ 処理時間と注意事項
- Colab CPU で銘柄あたり 2〜4 分目安（GPU で大幅短縮）
- 本予測は教育・学習目的です。投資判断は自己責任で行ってください

In [ ]:
# ===== AI予測（Stage 3）パラメータ =====

RUN_NN_PREDICTION  = True    # False にするとAI予測をスキップ
NN_MODELS          = ['LSTM-Simple', 'LSTM-Stacked', 'GRU', 'Transformer']  # 比較するモデル
NN_USE_FEATURES    = True    # True: 5特徴量（多変量）、False: 終値log収益率のみ
NN_MC_SAMPLES      = 20      # MC Dropout サンプリング回数（不確実性バンド）
NN_BAND_PCT        = (10, 90)  # 予測区間のパーセンタイル（下限, 上限）
NN_TOP_K           = 3       # Stage 2 上位 何銘柄に対して予測を実行するか
NN_LOOK_BACK       = 60      # 入力シーケンス長（過去何日で予測するか）
NN_TRAIN_YEARS     = 1.5     # 訓練に使う過去年数（1.5年 = 直近データ重視）
NN_EPOCHS          = 30      # 訓練エポック数（EarlyStopping=5 で早期終了）
NN_BATCH           = 32      # バッチサイズ
NN_FORECAST_DAYS   = 30      # 何日先まで予測するか（反復予測）

if RUN_NN_PREDICTION:
    print('AI予測（Stage 3）設定:')
    print(f'  モデル       : {", ".join(NN_MODELS)}')
    feat_desc = '多変量 (close_logret / vol / volatility / rsi / sma20_dist)' if NN_USE_FEATURES else '単変量 (close_logret のみ)'
    print(f'  入力特徴量   : {feat_desc}')
    print(f'  MCサンプル数 : {NN_MC_SAMPLES}  予測区間: {NN_BAND_PCT[0]}〜{NN_BAND_PCT[1]} パーセンタイル')
    print(f'  対象銘柄数   : Stage 2 上位 {NN_TOP_K} 銘柄')
    print(f'  訓練期間     : 過去 {NN_TRAIN_YEARS} 年分')
    print(f'  予測         : {NN_FORECAST_DAYS} 日先（不確実性バンド付き）')
    print(f'  訓練設定     : {NN_EPOCHS} エポック, EarlyStopping(patience=5)')
else:
    print('AI予測（Stage 3）は無効化されています。RUN_NN_PREDICTION = True で有効化できます。')

# --- 狙い目ハントモード ---
NN_HUNT_MODE = True   # True: 「今が買い時」エントリー候補もAIで追加分析
NN_HUNT_K    = 5      # 狙い目候補から何銘柄をAI分析するか
if RUN_NN_PREDICTION and NN_HUNT_MODE:
    print(f'  狙い目ハント : 有効（エントリー候補 上位 {NN_HUNT_K} 銘柄を追加）')

In [ ]:
# ===== モデル定義と学習・評価関数（対数収益率ベース）=====
if RUN_NN_PREDICTION:
    import os
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

    import tensorflow as tf
    from tensorflow.keras.models import Sequential, Model
    from tensorflow.keras.layers import (LSTM, GRU, Dense, Dropout,
        MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D, Input, Add)
    from tensorflow.keras.callbacks import EarlyStopping

    print(f'✓ TensorFlow {tf.__version__} 読み込み完了')

    # ---- 特徴量エンジニアリング（多変量モード）----
    def _build_features(close_arr, volume_arr, rsi_series, train_size):
        """5特徴量行列を構築してz-score正規化。"""
        close_logret = np.diff(np.log(np.maximum(close_arr, 1e-10)))

        vol = np.where(volume_arr > 0, volume_arr.astype(float), 1.0)
        vol_logret = np.diff(np.log(vol))

        volatility = np.array([
            np.std(close_logret[max(0, i - 9):i + 1]) for i in range(len(close_logret))
        ])

        rsi_vals = rsi_series.fillna(50.0).values
        rsi_norm = rsi_vals[1:] / 50.0 - 1.0  # [0,100] -> [-1,1]

        n = len(close_arr)
        sma20 = np.array([close_arr[max(0, i - 19):i + 1].mean() for i in range(n)])
        sma20_dist = (close_arr / (sma20 + 1e-10) - 1.0)[1:]

        feat = np.column_stack([close_logret, vol_logret, volatility, rsi_norm, sma20_dist])
        feat_means = feat[:train_size].mean(axis=0)
        feat_stds  = feat[:train_size].std(axis=0) + 1e-8
        feat_scaled = (feat - feat_means) / feat_stds

        return feat_scaled, feat_means, feat_stds, close_logret

    # ---- ナイーブ・ベースライン（ランダムウォーク比較）----
    def _naive_baseline(close_test, log_ret_test):
        """ランダムウォーク（前日価格据え置き）の MAPE と方向精度を返す。"""
        if len(close_test) < 2:
            return 5.0, 0.5
        # 価格予測 = 前日の実価格（ランダムウォーク）
        naive_pred = close_test[:-1]
        naive_mape = float(np.mean(np.abs((close_test[1:] - naive_pred) / (close_test[1:] + 1e-10))) * 100)
        # 方向予測 = 常に「変化なし（0）」
        dir_act = (log_ret_test > 0).astype(int)
        dir_prd = np.zeros(len(log_ret_test), dtype=int)
        naive_dir_acc = float(np.mean(dir_act == dir_prd))
        return naive_mape, naive_dir_acc

    # ---- 4種類のモデル定義（input_shape=(look_back, n_features)）----
    def build_lstm_simple(look_back, n_features=1):
        m = Sequential([
            LSTM(32, input_shape=(look_back, n_features)),
            Dropout(0.2), Dense(1),
        ])
        m.compile(optimizer='adam', loss='mse')
        return m

    def build_lstm_stacked(look_back, n_features=1):
        m = Sequential([
            LSTM(50, return_sequences=True, input_shape=(look_back, n_features)),
            Dropout(0.2),
            LSTM(50, return_sequences=True), Dropout(0.2),
            LSTM(50), Dropout(0.2), Dense(1),
        ])
        m.compile(optimizer='adam', loss='mse')
        return m

    def build_gru(look_back, n_features=1):
        m = Sequential([
            GRU(50, return_sequences=True, input_shape=(look_back, n_features)),
            Dropout(0.2), GRU(50), Dropout(0.2), Dense(1),
        ])
        m.compile(optimizer='adam', loss='mse')
        return m

    def build_transformer(look_back, n_features=1):
        d_model = 32
        inp = Input(shape=(look_back, n_features))
        x   = Dense(d_model)(inp)
        attn = MultiHeadAttention(num_heads=2, key_dim=d_model // 2, dropout=0.2)(x, x)
        x   = Add()([x, attn])
        x   = LayerNormalization()(x)
        x   = Dropout(0.2)(x)
        x   = GlobalAveragePooling1D()(x)
        x   = Dense(32, activation='relu')(x)
        x   = Dropout(0.2)(x)
        out = Dense(1)(x)
        m   = Model(inputs=inp, outputs=out)
        m.compile(optimizer='adam', loss='mse')
        return m

    _ALL_BUILDERS = {
        'LSTM-Simple':  build_lstm_simple,
        'LSTM-Stacked': build_lstm_stacked,
        'GRU':          build_gru,
        'Transformer':  build_transformer,
    }
    MODEL_BUILDERS = {k: v for k, v in _ALL_BUILDERS.items() if k in NN_MODELS}

    # ---- シーケンス生成（多変量対応）----
    def make_sequences(arr2d, look_back):
        """arr2d: (N, n_features) -> X: (N-look_back, look_back, n_features), y: (N-look_back,)"""
        X, y = [], []
        for i in range(look_back, len(arr2d)):
            X.append(arr2d[i - look_back:i, :])
            y.append(arr2d[i, 0])   # target = feature 0 (close_logret)
        return np.array(X), np.array(y)

    # ---- 1モデルの訓練・評価 ----
    def train_and_evaluate_logrtn(builder_fn, X_tr, y_tr, X_te, y_te,
                                   log_ret_test, start_price_test,
                                   close_test, ret_scale, epochs, batch):
        tf.random.set_seed(42)
        look_back, n_features = X_tr.shape[1], X_tr.shape[2]
        model = builder_fn(look_back, n_features)
        es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        model.fit(X_tr, y_tr, epochs=epochs, batch_size=batch,
                  validation_data=(X_te, y_te), callbacks=[es], verbose=0)

        pred_scaled  = model.predict(X_te, verbose=0).flatten()
        pred_log_ret = pred_scaled * ret_scale
        pred_prices  = start_price_test * np.exp(np.cumsum(pred_log_ret))

        n      = min(len(close_test), len(pred_prices))
        actual = close_test[:n]
        pred   = pred_prices[:n]

        mape    = float(np.mean(np.abs((actual - pred) / (actual + 1e-10))) * 100)
        dir_act = (log_ret_test[:n] > 0).astype(int)
        dir_prd = (pred_log_ret[:n]  > 0).astype(int)
        dir_acc = float(np.mean(dir_act == dir_prd))
        composite = mape / (dir_acc + 0.01)

        return model, pred_prices, mape, dir_acc, composite

    # ---- MC Dropout 反復予測（不確実性バンド生成）----
    def _mc_forecast(model, last_seq, ret_scale, current_price, last_close_window,
                      forecast_days, mc_samples, band_pct,
                      feat_means=None, feat_stds=None):
        """
        MC Dropout で forecast_days 日分の価格パスを mc_samples 本生成する。
        close由来特徴（feat 0,2,4）は予測価格から再計算、
        外生特徴（feat 1: vol_logret, feat 3: rsi_norm）は最終観測値を据え置く。
        """
        look_back, n_features = last_seq.shape
        all_paths = []

        for _ in range(mc_samples):
            seq          = last_seq.copy()
            log_rets     = []
            price_window = list(last_close_window[-20:])

            for _step in range(forecast_days):
                inp        = seq.reshape(1, look_back, n_features)
                pred_sc    = float(model(inp, training=True)[0, 0])   # MC Dropout
                actual_ret = pred_sc * ret_scale
                log_rets.append(actual_ret)

                next_px = price_window[-1] * np.exp(actual_ret)
                price_window.append(next_px)
                if len(price_window) > 20:
                    price_window.pop(0)

                if n_features == 1:
                    next_row = np.array([pred_sc])
                else:
                    next_row = seq[-1].copy()
                    next_row[0] = pred_sc   # feat 0: close_logret (scaled)
                    # feat 1: vol_logret — 据え置き（外生）
                    # feat 2: volatility — 直近 log_ret から再計算
                    recent_rets = [seq[j, 0] * ret_scale for j in range(-min(10, look_back), 0)]
                    recent_rets.append(actual_ret)
                    vol_raw = float(np.std(recent_rets))
                    if feat_stds is not None and feat_stds[2] > 0:
                        next_row[2] = (vol_raw - feat_means[2]) / feat_stds[2]
                    # feat 3: rsi_norm — 中立に向けて緩やかに減衰（外生近似）
                    next_row[3] = seq[-1, 3] * 0.95
                    # feat 4: sma20_dist — 予測価格の SMA20 から再計算
                    sma20_val = float(np.mean(price_window[-min(20, len(price_window)):]))
                    sma20_raw = price_window[-1] / (sma20_val + 1e-10) - 1.0
                    if feat_stds is not None and feat_stds[4] > 0:
                        next_row[4] = (sma20_raw - feat_means[4]) / feat_stds[4]

                seq = np.vstack([seq[1:], next_row])

            path = current_price * np.exp(np.cumsum(log_rets))
            all_paths.append(path)

        arr = np.array(all_paths)   # (mc_samples, forecast_days)
        return (np.median(arr, axis=0),
                np.percentile(arr, band_pct[0], axis=0),
                np.percentile(arr, band_pct[1], axis=0),
                arr)

    # ---- 1銘柄の全工程: 取得 → 前処理 → 訓練 → 最適選択 → MC 予測 ----
    def predict_with_best_model(ticker, name, train_years, look_back,
                                 forecast_days, epochs, batch):
        from datetime import datetime as _dt, timedelta as _td
        end   = _dt.today()
        start = end - _td(days=int(train_years * 365) + 60)
        try:
            df = yf.download(ticker,
                             start=start.strftime('%Y-%m-%d'),
                             end=end.strftime('%Y-%m-%d'),
                             progress=False, auto_adjust=False)
        except Exception:
            return None
        if df is None or df.empty:
            return None
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if 'Close' not in df.columns or len(df) < look_back + 80:
            return None

        close_arr  = df['Close'].astype(float).values
        volume_arr = (df['Volume'].astype(float).values
                      if 'Volume' in df.columns else np.ones(len(close_arr)))

        if NN_USE_FEATURES:
            rsi_series = compute_rsi(pd.Series(close_arr), 14)
            N_log      = len(close_arr) - 1
            train_size = int(N_log * 0.8)
            if train_size < look_back + 20:
                return None
            feat_scaled, feat_means, feat_stds, close_logret = _build_features(
                close_arr, volume_arr, rsi_series, train_size)
            n_features  = 5
            ret_scale   = float(feat_stds[0])   # std of close_logret in training
            data_matrix = feat_scaled            # (N_log, 5)
        else:
            close_logret = np.diff(np.log(np.maximum(close_arr, 1e-10)))
            N_log        = len(close_logret)
            train_size   = int(N_log * 0.8)
            if train_size < look_back + 20:
                return None
            ret_std     = float(np.std(close_logret[:train_size])) + 1e-8
            ret_scale   = 3.0 * ret_std
            data_matrix = (close_logret / ret_scale).reshape(-1, 1)
            feat_means, feat_stds, n_features = None, None, 1

        X_train, y_train = make_sequences(data_matrix[:train_size], look_back)
        X_test,  y_test  = make_sequences(data_matrix[train_size - look_back:], look_back)

        start_price_test = float(close_arr[train_size])
        close_test       = close_arr[train_size + 1:]
        log_ret_test     = close_logret[train_size:]

        naive_mape, naive_dir_acc = _naive_baseline(close_test, log_ret_test)

        per_model = {}
        best = {'name': None, 'comp': float('inf'), 'mape': None,
                'dir_acc': None, 'model': None, 'pred': None}

        for mname, builder in MODEL_BUILDERS.items():
            try:
                m, pred_p, mape, dir_acc, comp = train_and_evaluate_logrtn(
                    builder, X_train, y_train, X_test, y_test,
                    log_ret_test, start_price_test, close_test,
                    ret_scale, epochs, batch)
                per_model[mname] = {'mape': mape, 'dir_acc': dir_acc, 'composite': comp}
                if comp < best['comp']:
                    best = {'name': mname, 'comp': comp, 'mape': mape,
                            'dir_acc': dir_acc, 'model': m, 'pred': pred_p}
            except Exception as e:
                per_model[mname] = {'mape': None, 'dir_acc': None,
                                    'composite': None, 'error': str(e)}

        if best['model'] is None:
            return None

        beats_naive = bool(best['mape'] < naive_mape)

        # MC Dropout で将来予測 + 不確実性バンド
        last_seq          = data_matrix[-look_back:].reshape(look_back, n_features)
        last_close_window = close_arr[-20:]
        current_price     = float(close_arr[-1])

        forecast_median, forecast_lower, forecast_upper, mc_paths = _mc_forecast(
            best['model'], last_seq, ret_scale, current_price, last_close_window,
            forecast_days, NN_MC_SAMPLES, NN_BAND_PCT,
            feat_means if NN_USE_FEATURES else None,
            feat_stds  if NN_USE_FEATURES else None)

        forecast_end = float(forecast_median[-1])
        forecast_ret = (forecast_end / current_price - 1) * 100

        ai_signal    = float(np.tanh(forecast_ret / 5.0))
        ai_confidence = 1.0 / (best['mape'] + 1.0)
        ai_score      = ai_signal * ai_confidence * 100

        n_pred     = min(len(close_test), len(best['pred']))
        test_dates = df.index[train_size + 1: train_size + 1 + n_pred]

        return {
            'ticker': ticker, 'name': name,
            'best_model':     best['name'],
            'best_mape':      best['mape'],
            'best_dir_acc':   best['dir_acc'],
            'best_composite': best['comp'],
            'per_model':      per_model,
            'current_price':  current_price,
            'forecast_end_price':   forecast_end,
            'forecast_lower_price': float(forecast_lower[-1]),
            'forecast_upper_price': float(forecast_upper[-1]),
            'forecast_return':      forecast_ret,
            'ai_signal':      ai_signal,
            'ai_confidence':  ai_confidence,
            'ai_score':       ai_score,
            'naive_mape':     naive_mape,
            'naive_dir_acc':  naive_dir_acc,
            'beats_naive':    beats_naive,
            'forecast_dates':  pd.date_range(df.index[-1] + pd.Timedelta(days=1),
                                              periods=forecast_days),
            'forecast_median': forecast_median,
            'forecast_lower':  forecast_lower,
            'forecast_upper':  forecast_upper,
            'history_dates':   df.index,
            'history_prices':  close_arr,
            'test_dates':      test_dates,
            'test_actual':     close_test[:n_pred],
            'test_pred':       best['pred'][:n_pred],
        }

    feat_info = '多変量5特徴' if NN_USE_FEATURES else '単変量'
    print(f'✓ モデル定義完了（{feat_info} / {list(MODEL_BUILDERS.keys())} / MC={NN_MC_SAMPLES}）')

In [ ]:
# ===== Stage 3 実行: テクニカル上位 + 狙い目候補のAI予測 =====
if RUN_NN_PREDICTION:
    stage3_start = time.time()

    # テクニカル上位銘柄
    tech_targets = [(row['ticker'], row['name'], 'テクニカル上位')
                    for _, row in ranking_df.head(NN_TOP_K).iterrows()]

    # 狙い目ハント候補（テクニカル上位と重複しないもの）
    hunt_targets = []
    if NN_HUNT_MODE and not entry_df.empty:
        seen_tickers = {t[0] for t in tech_targets}
        for _, row in entry_df.head(NN_HUNT_K).iterrows():
            if row['ticker'] not in seen_tickers:
                hunt_targets.append((row['ticker'], row['name'], '🎯狙い目'))

    ai_targets = tech_targets + hunt_targets
    print(f'\nStage 3 開始: テクニカル上位 {len(tech_targets)} + '
          f'狙い目 {len(hunt_targets)} = 計 {len(ai_targets)} 銘柄\n')
    feat_info = '多変量5特徴' if NN_USE_FEATURES else '単変量'
    print(f'  ※ 入力: {feat_info}   ※ 選択基準: MAPE÷(方向精度+ε) 最小化   ※ 不確実性: MC Dropout {NN_MC_SAMPLES}サンプル\n')

    nn_results = []
    for j, (tkr, nm, src_label) in enumerate(ai_targets, 1):
        print(f'  [{j}/{len(ai_targets)}] [{src_label}] {tkr} ({nm}) を訓練中...',
              flush=True)
        r = predict_with_best_model(
            tkr, nm, NN_TRAIN_YEARS, NN_LOOK_BACK,
            NN_FORECAST_DAYS, NN_EPOCHS, NN_BATCH)
        if r is None:
            print('    [スキップ] データ不足または取得失敗')
            continue
        r['source'] = src_label
        nn_results.append(r)

        # モデル別成績表示
        lines = []
        for mn, v in r['per_model'].items():
            if v.get('mape') is not None:
                lines.append(f"{mn}: MAPE={v['mape']:.2f}% DirAcc={v['dir_acc']:.1%}")
            else:
                lines.append(f"{mn}: エラー")
        print('    ' + '  |  '.join(lines))

        # ナイーブ比較 + 予測区間
        beats_str = '✅NN有効' if r['beats_naive'] else '⚠️ナイーブ以下'
        print(f"    ナイーブ MAPE={r['naive_mape']:.2f}%  →  最適: {r['best_model']}  "
              f"MAPE={r['best_mape']:.2f}%  {beats_str}")
        print(f"    {NN_FORECAST_DAYS}日後 中央値 {r['forecast_end_price']:.1f} "
              f"(区間 {r['forecast_lower_price']:.1f}〜{r['forecast_upper_price']:.1f})")

    elapsed = time.time() - stage3_start
    print(f'\nStage 3 完了（{elapsed:.1f} 秒）')
else:
    nn_results = []
    print('Stage 3 はスキップされました（RUN_NN_PREDICTION = False）')

In [ ]:
# ===== AI予測結果の可視化・複合スコアランキング =====
if RUN_NN_PREDICTION and nn_results:

    # ---- 複合スコアの計算 ----
    tech_scores = ranking_df.set_index('ticker')['スコア'].to_dict()
    max_tech    = ranking_df['スコア'].max() + 1e-8

    for r in nn_results:
        ts = tech_scores.get(r['ticker'], 0)
        ts_norm = ts / max_tech * 100
        r['tech_score_raw']  = ts
        r['tech_score_norm'] = ts_norm
        r['combined_score']  = 0.6 * ts_norm + 0.4 * np.clip(r['ai_score'], -50, 50)
        tech_up = ts >= 40
        ai_up   = r['forecast_return'] > 0
        r['signal'] = ('🟢 両シグナル↑' if tech_up and ai_up
                       else ('🟡 テクニカル↑/AI↓' if tech_up
                             else ('🟠 AI↑/テクニカル低' if ai_up
                                   else '🔴 両シグナル↓')))

    nn_results_sorted = sorted(nn_results, key=lambda x: x['combined_score'], reverse=True)

    # ---- 複合ランキング表示 ----
    print('=' * 85)
    print('🏆 複合スコアランキング  （テクニカル 60% ＋ AI 40%）')
    print('=' * 85)
    print(f"{'順位':<4} {'経路':<6} {'銘柄':^18} {'複合':>6} {'テク':>6} {'AIスコア':>8} "
          f"{'AI予測%':>8} {'MAPE':>6} {'ナイーブ':>8} {'NN有効':>6} {'シグナル'}")
    print('-' * 90)
    for k, r in enumerate(nn_results_sorted, 1):
        src_lbl   = r.get('source', 'テク上位')[:5]
        beats_str = '✅' if r['beats_naive'] else '⚠️'
        print(f"{k:<4} {src_lbl:<6} {r['ticker']:>8} {r['name'][:8]:<8} "
              f"{r['combined_score']:>6.1f} {r['tech_score_norm']:>6.1f} "
              f"{r['ai_score']:>+8.1f} {r['forecast_return']:>+8.2f}% "
              f"{r['best_mape']:>5.2f}% {r['naive_mape']:>7.2f}% "
              f"  {beats_str}   {r['signal']}")
    print()
    print('  🟢 両シグナル↑ : テクニカル・AI ともに上昇シグナル → 最も強い推奨')
    print('  🟡 テクニカル↑ : 過去の株価推移は良好だが AI は慎重')
    print('  🟠 AI↑のみ    : AI が上昇を見込むがテクニカルは低評価')
    print('  🔴 両シグナル↓ : テクニカル・AI ともに弱い → 見送り推奨')
    print('  ✅NN有効       : NNがランダムウォーク基準（ナイーブ）を下回るMAPEを達成')
    print('  ⚠️ナイーブ以下  : NN精度がナイーブ未満 → 予測の信頼度が低い')

    # ---- CSV 出力 ----
    nn_df = pd.DataFrame([
        {
            'ticker': r['ticker'], 'name': r['name'],
            'AI最適モデル':          r['best_model'],
            'AI_MAPE%':              round(r['best_mape'], 2),
            'ナイーブMAPE%':         round(r['naive_mape'], 2),
            'NN有効':                r['beats_naive'],
            '方向精度':              round(r['best_dir_acc'] * 100, 1),
            '現在価格':              round(r['current_price'], 2),
            f'{NN_FORECAST_DAYS}日後予測(中央値)': round(r['forecast_end_price'], 2),
            '予測下限':              round(r['forecast_lower_price'], 2),
            '予測上限':              round(r['forecast_upper_price'], 2),
            'AI予測リターン%':       round(r['forecast_return'], 2),
            'AI信頼スコア':          round(r['ai_score'], 1),
            '複合スコア':            round(r['combined_score'], 1),
            'シグナル':              r['signal'],
            'AI分析経路':            r.get('source', 'テクニカル上位'),
        }
        for r in nn_results_sorted
    ])
    augmented_ranking = ranking_df.merge(nn_df, on=['ticker', 'name'], how='left')

    aug_filename = f"recommend_with_ai_{datetime.today().strftime('%Y%m%d')}.csv"
    try:
        augmented_ranking.to_csv(aug_filename, encoding='utf-8-sig')
        print(f'\n✓ AI予測込みランキングを保存: {aug_filename}')
        try:
            from google.colab import files as colab_files
            colab_files.download(aug_filename)
        except Exception:
            pass
    except Exception as e:
        print(f'⚠️ CSV保存に失敗: {e}')

    # ---- 可視化 1: 銘柄ごとの予測チャート（不確実性バンド付き）----
    n   = len(nn_results_sorted)
    fig, axes = plt.subplots(n, 1, figsize=(14, 4.5 * n))
    if n == 1:
        axes = [axes]

    for ax, r in zip(axes, nn_results_sorted):
        recent = min(180, len(r['history_dates']))
        ax.plot(r['history_dates'][-recent:], r['history_prices'][-recent:],
                label='実績株価', color='#1565C0', linewidth=2)
        if len(r['test_dates']) > 0 and len(r['test_pred']) > 0:
            ax.plot(r['test_dates'], r['test_pred'],
                    label=f"{r['best_model']} テスト予測",
                    color='#FF8F00', linewidth=1.4, linestyle='--', alpha=0.85)

        # 中央値予測線
        ax.plot(r['forecast_dates'], r['forecast_median'],
                label=f"{NN_FORECAST_DAYS}日先 中央値 ({r['forecast_return']:+.1f}%)",
                color='#C62828', linewidth=2.0, marker='o', markersize=3)
        # 不確実性バンド（fill_between）
        ax.fill_between(r['forecast_dates'], r['forecast_lower'], r['forecast_upper'],
                        color='#EF9A9A', alpha=0.35,
                        label=f"予測区間 {NN_BAND_PCT[0]}〜{NN_BAND_PCT[1]}%ile")
        # ナイーブ基準線（現在価格の水平線）
        ax.axhline(y=r['current_price'], color='#78909C',
                   linestyle='-.', linewidth=1.0, alpha=0.7, label='現在価格（ナイーブ基準）')

        ax.axvline(x=r['history_dates'][-1], color='#2E7D32',
                   linestyle=':', linewidth=1.2, alpha=0.7)
        ymax = ax.get_ylim()[1]
        ax.text(r['history_dates'][-1], ymax * 0.98, ' 現在',
                color='#2E7D32', fontsize=9, va='top')

        beats_str = '✅NN有効' if r['beats_naive'] else '⚠️ナイーブ以下'
        ax.set_title(
            f"{r.get('source','テク')} | {r['ticker']} ({r['name']})  "
            f"最適:{r['best_model']}  MAPE:{r['best_mape']:.2f}%  "
            f"ナイーブMAPE:{r['naive_mape']:.2f}%  {beats_str}  {r['signal']}",
            fontsize=10)
        ax.set_ylabel('価格')
        ax.legend(loc='upper left', fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    ai_chart_file = 'ai_forecast.png'
    plt.savefig(ai_chart_file, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'✓ 予測チャートを保存: {ai_chart_file}')

    # ---- 可視化 2: 複合スコアと各シグナルの比較バーチャート ----
    fig2, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(14, max(3, n * 0.9 + 1)))

    labels   = [f"{r['ticker']}\n{r['name']}" for r in nn_results_sorted]
    y_pos    = np.arange(len(nn_results_sorted))
    bar_h    = 0.3

    tech_vals = [r['tech_score_norm'] for r in nn_results_sorted]
    ai_vals   = [np.clip(r['ai_score'], -50, 50) for r in nn_results_sorted]
    comb_vals = [r['combined_score']  for r in nn_results_sorted]

    ax_left.barh(y_pos + bar_h, tech_vals, bar_h, label='テクニカルスコア', color='#42A5F5')
    ax_left.barh(y_pos, ai_vals, bar_h, label='AI信頼スコア',
                 color=[('#26A69A' if v >= 0 else '#EF5350') for v in ai_vals])
    ax_left.axvline(0, color='black', linewidth=0.8)
    ax_left.set_yticks(y_pos + bar_h / 2)
    ax_left.set_yticklabels(labels, fontsize=9)
    ax_left.set_xlabel('スコア')
    ax_left.set_title('テクニカル vs AI シグナル', fontsize=11)
    ax_left.legend(fontsize=9)
    ax_left.grid(axis='x', alpha=0.3)
    ax_left.invert_yaxis()

    colors_comb = [('#1565C0' if v >= 40 else '#90A4AE') for v in comb_vals]
    hbars = ax_right.barh(labels, comb_vals, color=colors_comb, height=0.5)
    for bar, r in zip(hbars, nn_results_sorted):
        beats_str = '✅' if r['beats_naive'] else '⚠️'
        ax_right.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                      f"{r['combined_score']:.1f}  {beats_str}  {r['signal']}",
                      va='center', fontsize=8)
    ax_right.set_xlabel('複合スコア（テクニカル 60% ＋ AI 40%）')
    ax_right.set_title('最終複合スコアランキング', fontsize=11)
    ax_right.set_xlim(0, max(comb_vals) * 1.45 if comb_vals else 10)
    ax_right.grid(axis='x', alpha=0.3)
    ax_right.invert_yaxis()

    plt.tight_layout()
    plt.savefig('ai_combined_ranking.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('✓ 複合ランキングチャートを保存: ai_combined_ranking.png')

    # ---- 可視化 3: モデル別 MAPE + 方向精度ヒートマップ ----
    model_names = list(MODEL_BUILDERS.keys())
    mape_mat = np.full((len(nn_results_sorted), len(model_names)), np.nan)
    dir_mat  = np.full((len(nn_results_sorted), len(model_names)), np.nan)
    for i, r in enumerate(nn_results_sorted):
        for j, mn in enumerate(model_names):
            v = r['per_model'].get(mn, {})
            if v.get('mape') is not None:
                mape_mat[i, j] = v['mape']
                dir_mat[i,  j] = v['dir_acc'] * 100

    fig3, (ax_m, ax_d) = plt.subplots(1, 2, figsize=(12, max(3, n * 0.8 + 1)))
    for ax_h, mat, cmap_h, title_h, fmt_h in [
        (ax_m, mape_mat, 'RdYlGn_r', 'MAPE (%) — 低いほど良', '{:.1f}%'),
        (ax_d, dir_mat,  'RdYlGn',   '方向精度 (%) — 高いほど良', '{:.0f}%'),
    ]:
        masked = np.ma.masked_invalid(mat)
        im = ax_h.imshow(masked, cmap=cmap_h, aspect='auto')
        ax_h.set_xticks(range(len(model_names)))
        ax_h.set_xticklabels(model_names, fontsize=9)
        ax_h.set_yticks(range(len(nn_results_sorted)))
        ax_h.set_yticklabels([r['ticker'] for r in nn_results_sorted], fontsize=9)
        ax_h.set_title(title_h, fontsize=11)
        plt.colorbar(im, ax=ax_h, shrink=0.8)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                if not np.isnan(mat[i, j]):
                    ax_h.text(j, i, fmt_h.format(mat[i, j]),
                              ha='center', va='center', fontsize=9, color='black')
        for i, r in enumerate(nn_results_sorted):
            if r['best_model'] in model_names:
                j = model_names.index(r['best_model'])
                ax_h.text(j, i - 0.3, '⭐', ha='center', va='bottom', fontsize=11)

    plt.tight_layout()
    plt.savefig('ai_model_heatmap.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('✓ モデル精度ヒートマップを保存: ai_model_heatmap.png')

    print('\n⚠️ 免責事項: AIによる予測は過去パターンに基づくものであり、将来の株価を保証しません。')
    print('   投資判断はご自身の責任で行ってください。')

elif RUN_NN_PREDICTION:
    print('⚠️ AI予測結果がありません（全銘柄でデータ不足）')

---
## 9. バックテスト検証（Stage 4）— このテクニカル戦略は1ヶ月で利益が出るか？

Stage 2 のテクニカルスコアが**実際に将来リターンと結びつくか**を、過去データで検証します。

### やること
1. 対象銘柄ごとに過去データを取得し、**各時点で利用可能なデータだけ**を使ってテクニカルスコアを計算（先読みバイアスなし）
2. その時点から **N営業日後（既定30日 ≒ 1ヶ月）のリターン**を測定
3. 「スコアが高い時に買う」戦略の **勝率・平均リターン・最大損失** を集計
4. **スコアと将来リターンの相関**を可視化し、スコアが本当に効いているかを確認

### 読み方
- **買いシグナル（スコア≥閾値）の勝率 > 50% かつ 平均リターン > 全体平均** なら、テクニカルスコアに優位性（エッジ）あり
- **エッジ ≒ 0 または マイナス** なら、現在の選定ロジックは1ヶ月利益の根拠として弱い → 閾値やスコア配点の見直しが必要
- **最大損失** の大きさで、損切りルールの必要性が分かる

> ⚠️ これは「過去にこのルールで買っていたらどうだったか」のシミュレーションです。
> 売買コスト・スリッページ・税金は含みません。将来の成績を保証するものではありません。

In [ ]:
# ===== Stage 4: バックテスト検証（テクニカルスコア vs 30日後リターン）=====
import time as _time_bt

RUN_BACKTEST            = True   # False で本セルをスキップ
BACKTEST_HOLD_DAYS      = 30     # 何営業日後のリターンを評価するか（≒1ヶ月）
BACKTEST_HISTORY_YEARS  = 2      # 取得する過去データの年数
BACKTEST_STEP_DAYS      = 5      # 何営業日ごとにエントリー判定をサンプリングするか
BACKTEST_MAX_TICKERS    = 30     # バックテスト対象銘柄数の上限（処理時間の制御）
BACKTEST_SCORE_THRESHOLD = 50    # 「買いシグナル」とみなすテクニカルスコアの閾値

if RUN_BACKTEST:
    bt_start = _time_bt.time()

    # ---- 対象銘柄（Stage 2 のランキング上位を流用）----
    bt_universe = [(row['ticker'], row['name'])
                   for _, row in ranking_df.head(BACKTEST_MAX_TICKERS).iterrows()]
    print(f'バックテスト対象: {len(bt_universe)} 銘柄  '
          f'（保有 {BACKTEST_HOLD_DAYS} 営業日 / {BACKTEST_STEP_DAYS} 日おきにエントリー判定）\n')

    # ---- 各時点のテクニカルスコアを因果的に計算するヘルパー ----
    # Cell 22 の analyze_ticker と同じ配点。指標は全期間で一度だけ計算し、
    # 時点 t では t までの値のみ参照する（rolling は過去のみ使うので先読みなし）。
    def _score_at(t, close, volume, sma20, sma50, sma200, rsi14, bb_mid, bb_std):
        c = float(close[t])
        s20 = sma20[t]; s50 = sma50[t]; s200 = sma200[t]; r = rsi14[t]
        if np.isnan(s20) or np.isnan(s50) or np.isnan(s200) or np.isnan(r):
            return None
        score = 0.0
        # 1. トレンド
        if c > s20:   score += 10
        if s20 > s50: score += 10
        if s50 > s200: score += 10
        # 2. ゴールデンクロス（直近10営業日以内）
        if t >= 11:
            a = sma20[t-10:t+1]; b = sma50[t-10:t+1]
            for i in range(1, len(a)):
                if (not np.isnan(a[i-1]) and not np.isnan(b[i-1])
                        and a[i-1] <= b[i-1] and a[i] > b[i]):
                    score += 15
                    break
        # 3. RSI
        if 30 <= r < 45:   score += 15
        elif 45 <= r <= 65: score += 10
        elif r > 75:        score -= 10
        # 4. モメンタム
        if t >= 20:
            ret20 = (c / float(close[t-20]) - 1) * 100
            score += max(min(ret20, 15), -10)
        if t >= 60:
            ret60 = (c / float(close[t-60]) - 1) * 100
            score += max(min(ret60 / 2, 10), -5)
        # 5. 出来高トレンド
        if t >= 19:
            vr = volume[t-4:t+1].mean() / (volume[t-19:t+1].mean() + 1e-10)
            if vr >= 1.5:   score += 10
            elif vr >= 1.1: score += 5
        # 6. ボリンジャーバンド位置
        if not np.isnan(bb_mid[t]) and not np.isnan(bb_std[t]) and bb_std[t] > 0:
            up = bb_mid[t] + 2 * bb_std[t]; lo = bb_mid[t] - 2 * bb_std[t]
            rng = up - lo
            if rng > 0:
                pos = (c - lo) / rng
                if 0.3 <= pos <= 0.7: score += 5
                elif pos > 1.0:       score -= 5
        return score

    # ---- 全銘柄でエントリー時点 → 30日後リターンを収集 ----
    samples = []   # (score, forward_ret_pct)
    end_bt   = datetime.today()
    start_bt = end_bt - timedelta(days=int(BACKTEST_HISTORY_YEARS * 365) + 260)
    n_ok = 0

    for idx, (tkr, nm) in enumerate(bt_universe, 1):
        try:
            df = yf.download(tkr, start=start_bt.strftime('%Y-%m-%d'),
                             end=end_bt.strftime('%Y-%m-%d'),
                             progress=False, auto_adjust=False)
        except Exception:
            continue
        if df is None or df.empty:
            continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if 'Close' not in df.columns or len(df) < 220 + BACKTEST_HOLD_DAYS:
            continue

        close_s  = df['Close'].astype(float)
        volume_s = (df['Volume'].astype(float) if 'Volume' in df.columns
                    else pd.Series(np.ones(len(close_s)), index=close_s.index))
        close  = close_s.values
        volume = volume_s.values
        sma20  = close_s.rolling(20).mean().values
        sma50  = close_s.rolling(50).mean().values
        sma200 = close_s.rolling(200).mean().values
        rsi14  = compute_rsi(close_s, 14).values
        bb_mid = close_s.rolling(20).mean().values
        bb_std = close_s.rolling(20).std().values

        last_valid = len(close) - BACKTEST_HOLD_DAYS
        for t in range(200, last_valid, BACKTEST_STEP_DAYS):
            sc = _score_at(t, close, volume, sma20, sma50, sma200, rsi14, bb_mid, bb_std)
            if sc is None:
                continue
            fwd = (float(close[t + BACKTEST_HOLD_DAYS]) / float(close[t]) - 1) * 100
            samples.append((sc, fwd))
        n_ok += 1
        if idx % 10 == 0 or idx == len(bt_universe):
            print(f'  [{idx}/{len(bt_universe)}] 収集済みサンプル: {len(samples)}')

    if not samples:
        print('\n⚠️ バックテスト用のデータが収集できませんでした。')
    else:
        arr   = np.array(samples)
        scores = arr[:, 0]
        fwds   = arr[:, 1]
        elapsed = _time_bt.time() - bt_start

        buy_mask  = scores >= BACKTEST_SCORE_THRESHOLD
        rest_mask = ~buy_mask

        def _stats(x):
            if len(x) == 0:
                return dict(n=0, win=np.nan, mean=np.nan, med=np.nan, mx=np.nan, mn=np.nan)
            return dict(n=len(x), win=float(np.mean(x > 0) * 100),
                        mean=float(np.mean(x)), med=float(np.median(x)),
                        mx=float(np.max(x)), mn=float(np.min(x)))

        all_st  = _stats(fwds)
        buy_st  = _stats(fwds[buy_mask])
        rest_st = _stats(fwds[rest_mask])
        corr    = float(np.corrcoef(scores, fwds)[0, 1]) if len(scores) > 2 else float('nan')
        edge    = (buy_st['mean'] - all_st['mean']) if buy_st['n'] > 0 else float('nan')

        print('\n' + '=' * 72)
        print(f'📊 バックテスト結果  （{n_ok} 銘柄 / {len(samples)} サンプル / {elapsed:.1f}秒）')
        print(f'   保有期間: {BACKTEST_HOLD_DAYS}営業日(≒1ヶ月)  '
              f'買い閾値: スコア≥{BACKTEST_SCORE_THRESHOLD}')
        print('=' * 72)
        print(f"{'区分':<22} {'件数':>6} {'勝率':>8} {'平均%':>8} {'中央%':>8} {'最大益%':>9} {'最大損%':>9}")
        print('-' * 72)
        for label, st in [('買いシグナル(高スコア)', buy_st),
                          ('非シグナル(低スコア)',   rest_st),
                          ('全サンプル(参考)',       all_st)]:
            if st['n'] > 0:
                print(f"{label:<22} {st['n']:>6} {st['win']:>7.1f}% "
                      f"{st['mean']:>+7.2f} {st['med']:>+7.2f} "
                      f"{st['mx']:>+8.1f} {st['mn']:>+8.1f}")
            else:
                print(f"{label:<22} {'0':>6}  （該当なし）")
        print('-' * 72)
        print(f"  スコアと{BACKTEST_HOLD_DAYS}日後リターンの相関係数: {corr:+.3f}  "
              f"（正で大きいほどスコアが有効）")
        print(f"  エッジ（買い平均 − 全体平均）: {edge:+.2f}%")

        # ---- 判定メッセージ ----
        if buy_st['n'] >= 10 and buy_st['win'] > 50 and edge > 0.5:
            verdict = '✅ テクニカルスコアに優位性あり。1ヶ月戦略の根拠として有効です。'
        elif buy_st['n'] >= 10 and (buy_st['win'] > 50 or edge > 0):
            verdict = '🟡 弱い優位性。閾値の引き上げや損切りルールで改善余地あり。'
        else:
            verdict = '🔴 明確な優位性は確認できず。スコア配点や保有期間の見直しを推奨。'
        print(f'\n  判定: {verdict}')
        if buy_st['n'] > 0 and buy_st['mn'] < -15:
            print(f'  ⚠️ 買いシグナルでも最大 {buy_st["mn"]:.1f}% の損失あり → 損切りルールの検討を推奨。')

        # ---- スコア帯別の集計表 ----
        bins   = [-100, 20, 40, 50, 60, 80, 1000]
        labels = ['<20', '20-40', '40-50', '50-60', '60-80', '80+']
        print('\n  スコア帯別の30日後リターン:')
        print(f"  {'スコア帯':<10} {'件数':>6} {'勝率':>8} {'平均%':>8}")
        bucket_mid, bucket_mean, bucket_win = [], [], []
        for bi in range(len(labels)):
            m = (scores >= bins[bi]) & (scores < bins[bi + 1])
            if m.sum() > 0:
                w = float(np.mean(fwds[m] > 0) * 100)
                mn = float(np.mean(fwds[m]))
                print(f"  {labels[bi]:<10} {int(m.sum()):>6} {w:>7.1f}% {mn:>+7.2f}")
                bucket_mid.append(labels[bi]); bucket_mean.append(mn); bucket_win.append(w)

        # ---- 可視化 ----
        fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 5))

        # 左: 散布図 + 回帰直線
        axL.scatter(scores, fwds, s=10, alpha=0.25, color='#1565C0')
        if len(scores) > 2:
            coef = np.polyfit(scores, fwds, 1)
            xs = np.linspace(scores.min(), scores.max(), 50)
            axL.plot(xs, np.polyval(coef, xs), color='#C62828', linewidth=2,
                     label=f'回帰直線 (相関 {corr:+.3f})')
        axL.axhline(0, color='black', linewidth=0.8)
        axL.axvline(BACKTEST_SCORE_THRESHOLD, color='#2E7D32', linestyle='--',
                    linewidth=1.2, label=f'買い閾値 {BACKTEST_SCORE_THRESHOLD}')
        axL.set_xlabel('テクニカルスコア（エントリー時点）')
        axL.set_ylabel(f'{BACKTEST_HOLD_DAYS}営業日後リターン (%)')
        axL.set_title('スコア vs 将来リターン', fontsize=12)
        axL.legend(fontsize=9)
        axL.grid(True, alpha=0.3)

        # 右: スコア帯別の平均リターン
        if bucket_mid:
            colors = ['#26A69A' if v >= 0 else '#EF5350' for v in bucket_mean]
            bars = axR.bar(bucket_mid, bucket_mean, color=colors)
            axR.axhline(0, color='black', linewidth=0.8)
            for b, w, mn in zip(bars, bucket_win, bucket_mean):
                axR.text(b.get_x() + b.get_width() / 2, b.get_height(),
                         f'勝率\n{w:.0f}%', ha='center',
                         va='bottom' if mn >= 0 else 'top', fontsize=8)
            axR.set_xlabel('テクニカルスコア帯')
            axR.set_ylabel(f'平均 {BACKTEST_HOLD_DAYS}日後リターン (%)')
            axR.set_title('スコア帯別の平均リターン', fontsize=12)
            axR.grid(axis='y', alpha=0.3)

        plt.tight_layout()
        plt.savefig('backtest_result.png', dpi=130, bbox_inches='tight')
        plt.show()
        print('\n✓ バックテスト結果を保存: backtest_result.png')
        print('⚠️ 売買コスト・スリッページ・税金は未考慮。将来成績を保証しません。')
else:
    print('バックテスト（Stage 4）はスキップされました（RUN_BACKTEST = False）')

---
## 10. 予測トラッキング（AI予測 vs 実績の比較）

Stage 3 で実行した AI 予測を**ログファイルに保存**し、後日改めて実行することで
**予測と実際の株価推移を比較**します。

### 仕組み
1. Stage 3 実行後にこのセルを実行すると、予測価格（中央値・予測区間）をログに追記
2. 翌日以降に再実行すると、実績の終値を自動取得してログに照合
3. 「追跡 MAPE」「方向精度」「予測区間内に収まった割合」を集計・可視化

### ログの保持方法
| 環境 | 保存先 | 備考 |
|---|---|---|
| **Google Drive マウント済み** | `/content/drive/MyDrive/stock_forecast_log.csv` | セッション跨ぎで自動引き継ぎ ✅ |
| **未マウント（ローカル）** | `stock_forecast_log.csv` | セル実行時に自動ダウンロードされるので保管しておく |

> Google Drive でのマウント方法:
> ```python
> from google.colab import drive
> drive.mount("/content/drive")
> ```
> マウント後にこのセルを再実行してください。

In [ ]:
# ===== Stage 5: 予測トラッキング（AI予測 vs 実績）=====
import os as _os

# ---- ログ保存先 ----
# Google Drive をマウントしてセッション間で履歴を引き継ぐ
_DRIVE_DIR   = '/content/drive/MyDrive'
MOUNT_DRIVE  = True   # Colab で Google Drive を自動マウントするか

# まだマウントされていなければマウントを試みる（Colab のみ）
if MOUNT_DRIVE and not _os.path.isdir(_DRIVE_DIR):
    try:
        from google.colab import drive as _gdrive
        print("Google Drive をマウントします（認証ダイアログが出たら許可してください）...")
        _gdrive.mount("/content/drive")
    except ModuleNotFoundError:
        pass   # Colab 以外の環境ではスキップ
    except Exception as _me:
        print(f"⚠️ Drive マウントに失敗しました ({_me})")

_DRIVE_AVAIL = _os.path.isdir(_DRIVE_DIR)
if _DRIVE_AVAIL:
    print(f"✓ Google Drive 利用可: {_DRIVE_DIR}")
else:
    print("ℹ️ Google Drive が未マウントです。履歴はこのセッション内のローカルファイルに保存されます。")
FORECAST_LOG = (f'{_DRIVE_DIR}/stock_forecast_log.csv'
                if _DRIVE_AVAIL else 'stock_forecast_log.csv')

RUN_TRACKING = True   # False でこのセルをスキップ

if RUN_TRACKING:
    _today = datetime.today()

    # ---------- (A) 既存ログを読み込む ----------
    log_df = pd.DataFrame()
    _log_exists  = _os.path.exists(FORECAST_LOG)
    _load_failed = False   # 安全装置: 既存ファイルがあるのに読めなかったか
    if _log_exists:
        try:
            log_df = pd.read_csv(FORECAST_LOG, encoding='utf-8-sig',
                                 parse_dates=['pred_run_date', 'forecast_date'])
            print(f'✓ 予測ログ読み込み: {len(log_df)} 行  ({FORECAST_LOG})')
        except Exception as _e:
            _load_failed = True
            print(f'⚠️ ログ読み込みエラー ({_e})')
            print('   既存ファイルが読めません。履歴保護のため今回は保存しません。')
    else:
        print('[情報] 予測ログが見つかりません。今回の予測から新規作成します。')
        if not _DRIVE_AVAIL:
            print('      セッション間で履歴を引き継ぐには Google Drive をマウントしてください。')

    # ---------- (B) 今回の予測をログに追記 ----------
    try:
        _nn = nn_results
    except NameError:
        _nn = []

    if _nn:
        _run_date_str = _today.strftime('%Y-%m-%d')
        _new_rows = []
        for _r in _nn:
            for _fd, _fm, _fl, _fu in zip(_r['forecast_dates'],
                                           _r['forecast_median'],
                                           _r['forecast_lower'],
                                           _r['forecast_upper']):
                _new_rows.append({
                    'pred_run_date':         _run_date_str,
                    'ticker':                _r['ticker'],
                    'name':                  _r['name'],
                    'forecast_date':         _fd.strftime('%Y-%m-%d'),
                    'current_price_at_pred': round(_r['current_price'], 4),
                    'forecast_median':       round(float(_fm), 4),
                    'forecast_lower':        round(float(_fl), 4),
                    'forecast_upper':        round(float(_fu), 4),
                    'best_model':            _r['best_model'],
                    'beats_naive':           _r['beats_naive'],
                    'actual_price':          float('nan'),
                    'actual_return_pct':     float('nan'),
                })
        _new_df = pd.DataFrame(_new_rows)
        _new_df['pred_run_date'] = pd.to_datetime(_new_df['pred_run_date'])
        _new_df['forecast_date'] = pd.to_datetime(_new_df['forecast_date'])

        # 同日・同銘柄の既存行は上書き
        if not log_df.empty:
            _dup = ((log_df['pred_run_date'].dt.normalize()
                     == pd.Timestamp(_run_date_str))
                    & (log_df['ticker'].isin(_new_df['ticker'].unique())))
            log_df = log_df[~_dup]

        log_df = pd.concat([log_df, _new_df], ignore_index=True)
        print(f'✓ {len(_new_df)} 行を追記 '
              f'（{len(_nn)} 銘柄 × {len(_new_df) // max(len(_nn),1)} 日分）')

    # ---------- (C) 過去の予測に実績価格を突き合わせ ----------
    if not log_df.empty:
        _fill_mask = ((log_df['forecast_date'] <=
                       pd.Timestamp(_today.date()))
                      & log_df['actual_price'].isna())
        _fill_rows = log_df[_fill_mask]

        if not _fill_rows.empty:
            print(f'\n実績価格を取得中（{_fill_rows["ticker"].nunique()} 銘柄）...')
            for _tkr in _fill_rows['ticker'].unique():
                _tr = _fill_rows[_fill_rows['ticker'] == _tkr]
                _fetch_start = (_tr['pred_run_date'].min()
                                - pd.Timedelta(days=5)).strftime('%Y-%m-%d')
                _fetch_end   = (_today + pd.Timedelta(days=2)).strftime('%Y-%m-%d')
                try:
                    _act = yf.download(_tkr, start=_fetch_start, end=_fetch_end,
                                        progress=False, auto_adjust=True)
                    if _act.empty: continue
                    if isinstance(_act.columns, pd.MultiIndex):
                        _act.columns = _act.columns.get_level_values(0)
                    _close = _act['Close'].astype(float)

                    for _idx in _tr.index:
                        _fd   = log_df.at[_idx, 'forecast_date']
                        _base = log_df.at[_idx, 'current_price_at_pred']
                        _fd_d = pd.Timestamp(_fd).date()
                        # 最も近い取引日（3営業日以内）を探す
                        _diffs = np.array([abs((pd.Timestamp(i).date() - _fd_d).days)
                                           for i in _close.index])
                        _min_d = _diffs.min()
                        if _min_d <= 3:
                            _px = float(_close.iloc[int(_diffs.argmin())])
                            log_df.at[_idx, 'actual_price']      = round(_px, 4)
                            log_df.at[_idx, 'actual_return_pct'] = round(
                                (_px / _base - 1) * 100, 4)
                except Exception:
                    pass
            print('✓ 突き合わせ完了')

    # ---------- (D) ログを保存（安全装置つき）----------
    # 安全装置1: 既存ファイルが読めなかった場合は上書きしない（履歴の事故消失を防ぐ）
    if _load_failed:
        print('\n🛡️ 履歴保護: 既存ログを読み込めなかったため保存をスキップしました。')
        print('   ファイル破損の可能性があります。バックアップを確認してください。')
        print(f'   問題なければ {FORECAST_LOG} を手動で退避・削除してから再実行してください。')
    elif log_df.empty:
        print('\n[情報] 保存対象のデータがありません。')
    else:
        try:
            # 安全装置2: 上書き前に既存ファイルを日時つきでバックアップ
            if _log_exists:
                import shutil as _shutil
                _bk_dir = (_DRIVE_DIR if _DRIVE_AVAIL else '.')
                _bk = f"{_bk_dir}/stock_forecast_log_backup_{_today.strftime('%Y%m%d_%H%M%S')}.csv"
                try:
                    _shutil.copy2(FORECAST_LOG, _bk)
                    print(f'🛡️ バックアップ作成: {_bk}')
                    # 古いバックアップは直近10件のみ保持
                    import glob as _glob
                    _bks = sorted(_glob.glob(f'{_bk_dir}/stock_forecast_log_backup_*.csv'))
                    for _old in _bks[:-10]:
                        try: _os.remove(_old)
                        except Exception: pass
                except Exception as _be:
                    print(f'⚠️ バックアップ作成に失敗 ({_be}) → 保存は続行します')

            log_df.to_csv(FORECAST_LOG, index=False, encoding='utf-8-sig')
            print(f'✓ 予測ログを保存: {len(log_df)} 行  ({FORECAST_LOG})')
            if not _DRIVE_AVAIL:
                try:
                    from google.colab import files as _cf
                    _cf.download(FORECAST_LOG)
                    print('  (ローカル保管用にダウンロード — 次回実行前にアップロードしてください)')
                except Exception:
                    pass
        except Exception as _e:
            print(f'⚠️ ログ保存エラー: {_e}')

    # ---------- (E) 集計・可視化 ----------
    _cmp = log_df[log_df['actual_price'].notna()].copy()

    if _cmp.empty:
        print('\n[情報] まだ実績価格と比較できるデータがありません。')
        print('  予測実行後、数営業日経ってから再実行すると比較が表示されます。')
    else:
        _cmp['pred_ret']  = (_cmp['forecast_median'] / _cmp['current_price_at_pred'] - 1) * 100
        _cmp['dir_ok']    = (_cmp['actual_return_pct'] > 0) == (_cmp['pred_ret'] > 0)
        _cmp['in_band']   = ((_cmp['actual_price'] >= _cmp['forecast_lower'])
                              & (_cmp['actual_price'] <= _cmp['forecast_upper']))
        _cmp['abs_err%']  = (np.abs(_cmp['actual_price'] - _cmp['forecast_median'])
                              / (_cmp['actual_price'] + 1e-10) * 100)

        # サマリーテーブル（予測日 × 銘柄）
        _sum = (_cmp.groupby(['pred_run_date', 'ticker', 'name'])
                .agg(比較日数=('actual_price', 'count'),
                     追跡MAPE=('abs_err%', 'mean'),
                     方向精度=('dir_ok', 'mean'),
                     バンド内率=('in_band', 'mean'))
                .reset_index())

        print('\n' + '=' * 76)
        print('📈 予測トラッキング結果（予測 vs 実績）')
        print('=' * 76)
        print(f"{'予測実行日':<12} {'銘柄':>9} {'名前':<14} {'比較日数':>6} "
              f"{'追跡MAPE':>9} {'方向精度':>8} {'バンド内率':>9}")
        print('-' * 76)
        for _, _row in _sum.iterrows():
            print(f"{str(_row['pred_run_date'])[:10]:<12}  {_row['ticker']:>8} "
                  f"{str(_row['name'])[:12]:<14} {int(_row['比較日数']):>6} "
                  f"  {_row['追跡MAPE']:>7.2f}%  {_row['方向精度']:>7.1%}  "
                  f"  {_row['バンド内率']:>7.1%}")
        print()
        print('  追跡MAPE   : 予測中央値と実績の平均絶対誤差率（小さいほど正確）')
        print('  方向精度   : 上昇↑/下落↓の方向が合っていた割合（50%=ランダム）')
        print('  バンド内率 : 実績が予測区間（10〜90%ile）に収まった割合（理論上 80% 目安）')

        # ---- 可視化: 予測 vs 実績 チャート ----
        _groups  = list(_cmp.groupby(['pred_run_date', 'ticker']))
        _ng      = len(_groups)
        _ncols   = min(2, _ng)
        _nrows   = (_ng + _ncols - 1) // _ncols

        fig_tr, _axes_2d = plt.subplots(_nrows, _ncols, squeeze=False,
                                         figsize=(13, 4.5 * _nrows))
        _axes_flat = _axes_2d.flatten()

        for _i, ((pdate, tkr), grp) in enumerate(_groups):
            _ax  = _axes_flat[_i]
            grp  = grp.sort_values('forecast_date')
            _nm  = grp['name'].iloc[0]
            _mod = grp['best_model'].iloc[0]
            _base = grp['current_price_at_pred'].iloc[0]

            # 予測区間（陰影）
            _ax.fill_between(grp['forecast_date'],
                              grp['forecast_lower'], grp['forecast_upper'],
                              color='#EF9A9A', alpha=0.35, label='予測区間')
            # 予測中央値
            _ax.plot(grp['forecast_date'], grp['forecast_median'],
                     color='#C62828', linewidth=1.8, linestyle='--',
                     label='予測中央値', zorder=3)
            # 実績株価
            _ax.plot(grp['forecast_date'], grp['actual_price'],
                     color='#1565C0', linewidth=2, marker='o', markersize=4,
                     label='実績株価', zorder=4)
            # 予測開始時の価格ライン
            _ax.axhline(_base, color='#555555', linewidth=0.8, linestyle=':',
                        alpha=0.7, label=f'予測開始価格 {_base:.1f}')

            _srow = _sum[(_sum['pred_run_date'] == pd.Timestamp(pdate))
                          & (_sum['ticker'] == tkr)]
            _m_str = (f"追跡MAPE:{_srow['追跡MAPE'].iloc[0]:.2f}%  "
                      f"方向:{_srow['方向精度'].iloc[0]:.1%}  "
                      f"バンド内:{_srow['バンド内率'].iloc[0]:.1%}"
                      if not _srow.empty else '')
            _ax.set_title(f"予測日:{str(pdate)[:10]}  {tkr} ({_nm})  {_mod}\n{_m_str}",
                          fontsize=9)
            _ax.set_ylabel('価格')
            _ax.legend(fontsize=8, loc='best')
            _ax.grid(True, alpha=0.3)

        for _ax in _axes_flat[_ng:]:
            _ax.set_visible(False)

        plt.tight_layout()
        plt.savefig('tracking_result.png', dpi=130, bbox_inches='tight')
        plt.show()
        print('✓ トラッキングチャートを保存: tracking_result.png')

else:
    print('予測トラッキングはスキップされました（RUN_TRACKING = False）')

### 9-2. ML モデル比較バックテスト

ルールベースの閾値判定と**機械学習モデル**の予測精度を同条件で比較します。

| モデル | 種別 | 特徴 |
|---|---|---|
| **ルールベース** | 閾値判定 | スコア ≥ 閾値を「買い」と判定 |
| **ロジスティック回帰** | 線形分類 | 特徴量の線形組み合わせ |
| **ランダムフォレスト** | 決定木アンサンブル | 過学習しにくい |
| **勾配ブースティング（sklearn）** | 決定木アンサンブル | 精度重視 |
| **LightGBM** | 勾配ブースティング | 高速・高精度 |

### 入力特徴量（9次元）
`close/SMA20`, `SMA20/SMA50`, `SMA50/SMA200`, `RSI14`, `BB位置`, `出来高比`, `20日リターン`, `60日リターン`, `テクニカルスコア`

### 評価方法
過去データを**時系列順に 70% 訓練 / 30% テスト**で分割（先読みなし）。
テスト期間で「買い予測銘柄の勝率」と「平均リターン」を比較します。

In [ ]:
# ===== Stage 4-ML: 機械学習モデル比較バックテスト =====
RUN_ML_BACKTEST = True   # False でスキップ

if RUN_ML_BACKTEST:
    try:
        _ = bt_universe   # Stage 4 バックテストが実行済みか確認
    except NameError:
        print('⚠️ Section 9（バックテスト）を先に実行してください。')
        raise

    # ---- ライブラリ ----
    try:
        import lightgbm as lgb
        _HAS_LGB = True
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'],
                       check=False)
        try:
            import lightgbm as lgb; _HAS_LGB = True
        except ImportError:
            _HAS_LGB = False
            print('⚠️ LightGBM インストールに失敗。LightGBM はスキップします。')

    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.preprocessing import StandardScaler

    # ---- 特徴量抽出関数 ----
    _ML_FEAT_NAMES = ['close_sma20', 'sma20_sma50', 'sma50_sma200',
                      'rsi14', 'bb_pos', 'vol_ratio', 'ret_20d', 'ret_60d', 'tech_score']

    def _features_at(t, close, volume, sma20, sma50, sma200, rsi14, bb_mid, bb_std, tech_score):
        if any(np.isnan(x) for x in [sma20[t], sma50[t], sma200[t], rsi14[t]]):
            return None
        c = float(close[t])
        close_sma20  = c / (sma20[t] + 1e-10) - 1
        sma20_sma50  = sma20[t] / (sma50[t] + 1e-10) - 1
        sma50_sma200 = sma50[t] / (sma200[t] + 1e-10) - 1
        rsi          = float(rsi14[t]) / 100.0   # 0〜1 に正規化
        bb_pos       = 0.5
        if not np.isnan(bb_std[t]) and bb_std[t] > 0:
            up = bb_mid[t] + 2 * bb_std[t]; lo = bb_mid[t] - 2 * bb_std[t]
            if up > lo: bb_pos = float(np.clip((c - lo) / (up - lo), 0, 1))
        vol_ratio    = (float(volume[t-4:t+1].mean()) / (float(volume[t-19:t+1].mean()) + 1e-10)
                        if t >= 19 else 1.0)
        ret_20d      = float(c / close[t-20] - 1) if t >= 20 else 0.0
        ret_60d      = float(c / close[t-60] - 1) if t >= 60 else 0.0
        return [close_sma20, sma20_sma50, sma50_sma200, rsi, bb_pos,
                vol_ratio, ret_20d, ret_60d, float(tech_score)]

    # ---- 特徴量収集ループ ----
    print(f'ML 特徴量を収集中（{len(bt_universe)} 銘柄 × {BACKTEST_HISTORY_YEARS} 年）...')
    ml_samples = []   # (feat_vec, fwd_ret, date_ordinal)

    for _mi, (tkr, nm) in enumerate(bt_universe, 1):
        try:
            _df = yf.download(tkr, start=start_bt.strftime('%Y-%m-%d'),
                               end=end_bt.strftime('%Y-%m-%d'),
                               progress=False, auto_adjust=False)
        except Exception:
            continue
        if _df is None or _df.empty: continue
        if isinstance(_df.columns, pd.MultiIndex):
            _df.columns = _df.columns.get_level_values(0)
        if 'Close' not in _df.columns or len(_df) < 220 + BACKTEST_HOLD_DAYS:
            continue

        _cs  = _df['Close'].astype(float)
        _vs  = (_df['Volume'].astype(float) if 'Volume' in _df.columns
                else pd.Series(np.ones(len(_cs)), index=_cs.index))
        _cl  = _cs.values; _vl = _vs.values
        _s20 = _cs.rolling(20).mean().values; _s50 = _cs.rolling(50).mean().values
        _s200= _cs.rolling(200).mean().values; _r14 = compute_rsi(_cs, 14).values
        _bm  = _cs.rolling(20).mean().values;  _bs  = _cs.rolling(20).std().values

        for _t in range(200, len(_cl) - BACKTEST_HOLD_DAYS, BACKTEST_STEP_DAYS):
            _sc = _score_at(_t, _cl, _vl, _s20, _s50, _s200, _r14, _bm, _bs)
            if _sc is None: continue
            _feat = _features_at(_t, _cl, _vl, _s20, _s50, _s200, _r14, _bm, _bs, _sc)
            if _feat is None: continue
            _fwd = (float(_cl[_t + BACKTEST_HOLD_DAYS]) / float(_cl[_t]) - 1) * 100
            _date_ord = _df.index[_t].toordinal()
            ml_samples.append((_feat, _fwd, _date_ord))

        if _mi % 10 == 0 or _mi == len(bt_universe):
            print(f'  [{_mi}/{len(bt_universe)}] サンプル: {len(ml_samples)}')

    if len(ml_samples) < 60:
        print('\n⚠️ サンプル数が少なすぎます（60 件未満）。ML 比較をスキップします。')
    else:
        # ---- 時系列順に分割 ----
        ml_samples.sort(key=lambda x: x[2])
        _split = int(len(ml_samples) * 0.70)

        X_all   = np.array([s[0] for s in ml_samples])
        y_ret   = np.array([s[1] for s in ml_samples])
        y_dir   = (y_ret > 0).astype(int)

        X_tr, X_te = X_all[:_split], X_all[_split:]
        y_tr_d, y_te_d = y_dir[:_split], y_dir[_split:]
        y_te_ret = y_ret[_split:]

        # スケーリング（ロジスティック回帰用）
        _scaler = StandardScaler()
        X_tr_sc = _scaler.fit_transform(X_tr)
        X_te_sc = _scaler.transform(X_te)

        # ---- モデル定義 ----
        _models_def = [
            (f'ルールベース(≥{BACKTEST_SCORE_THRESHOLD})',   None),
            ('ロジスティック回帰',    LogisticRegression(max_iter=500, random_state=42)),
            ('ランダムフォレスト',     RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)),
            ('勾配ブースティング',     GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42)),
        ]
        if _HAS_LGB:
            _models_def.append(('LightGBM',
                                 lgb.LGBMClassifier(n_estimators=200, max_depth=4,
                                                     random_state=42, verbose=-1)))

        # ---- 訓練・評価 ----
        ml_results = {}
        feat_imp_dict = {}
        print(f'\nモデル学習中（訓練 {_split} / テスト {len(y_te_d)} サンプル）...')

        for _mname, _model in _models_def:
            if _model is None:
                # ルールベース: テクニカルスコア（8番目のfeature=index8）を閾値判定
                _tech_scores_te = X_te[:, 8]
                _buy = _tech_scores_te >= BACKTEST_SCORE_THRESHOLD
            else:
                _Xf = X_tr_sc if 'ロジスティック' in _mname else X_tr
                _Xp = X_te_sc if 'ロジスティック' in _mname else X_te
                _model.fit(_Xf, y_tr_d)
                _buy = _model.predict(_Xp).astype(bool)
                if hasattr(_model, 'feature_importances_'):
                    feat_imp_dict[_mname] = _model.feature_importances_

            _buy_rets = y_te_ret[_buy]
            _win  = float(np.mean(_buy_rets > 0) * 100) if _buy.sum() > 0 else 0.0
            _mean = float(np.mean(_buy_rets))           if _buy.sum() > 0 else 0.0
            _med  = float(np.median(_buy_rets))         if _buy.sum() > 0 else 0.0
            _mn   = float(np.min(_buy_rets))            if _buy.sum() > 0 else 0.0
            ml_results[_mname] = dict(n=int(_buy.sum()), win=_win,
                                       mean=_mean, med=_med, worst=_mn)
            print(f'  ✓ {_mname}')

        # ---- 結果表示 ----
        _all_win  = float(np.mean(y_te_ret > 0) * 100)
        _all_mean = float(np.mean(y_te_ret))

        print('\n' + '=' * 80)
        print(f'🤖 ML モデル比較バックテスト（テスト: {len(y_te_d)} サンプル）')
        print(f'   全体の勝率: {_all_win:.1f}%  全体の平均リターン: {_all_mean:+.2f}%（ベースライン）')
        print('=' * 80)
        print(f"{'モデル':<26} {'シグナル':>6} {'勝率':>8} {'平均%':>8} {'中央%':>8} "
              f"{'最大損%':>9} {'エッジ':>8}")
        print('-' * 80)
        for _mn, _res in ml_results.items():
            _edge = _res['mean'] - _all_mean
            _flag = ('✅' if _res['win'] > 52 and _res['mean'] > _all_mean
                     else '🟡' if _res['win'] > 50 or _res['mean'] > _all_mean
                     else '🔴')
            print(f"{_flag} {_mn:<24} {_res['n']:>6} {_res['win']:>7.1f}% "
                  f"{_res['mean']:>+7.2f} {_res['med']:>+7.2f} "
                  f"{_res['worst']:>+8.1f} {_edge:>+7.2f}%")
        print()
        print('  エッジ = 買いシグナル平均リターン − 全体平均リターン（正 = ベースラインを上回る）')

        # ---- 可視化 ----
        _ng = len(ml_results)
        _has_fi = len(feat_imp_dict) > 0
        fig_ml, axes_ml = plt.subplots(1, 2 if _has_fi else 1,
                                        figsize=(14 if _has_fi else 7, 5))
        _ax1 = axes_ml[0] if _has_fi else axes_ml

        _mnames  = list(ml_results.keys())
        _wins    = [ml_results[m]['win']  for m in _mnames]
        _means   = [ml_results[m]['mean'] for m in _mnames]
        _y_pos   = np.arange(_ng)
        _bar_h   = 0.35

        _ax1.barh(_y_pos + _bar_h, _wins,   _bar_h,
                  color=['#26A69A' if v > 50 else '#EF5350' for v in _wins],
                  label='勝率 (%)')
        _ax1.barh(_y_pos, [m + 50 for m in _means], _bar_h,   # +50 で 0 基準に揃える
                  color=['#42A5F5' if v > 0 else '#EF5350' for v in _means],
                  label='平均リターン + 50 (%)')
        _ax1.axvline(50, color='black', linewidth=1.0, linestyle='--', label='50%基準')
        _ax1.axvline(_all_win, color='#FF8F00', linewidth=1.2, linestyle=':',
                     label=f'全体勝率 {_all_win:.1f}%')
        _ax1.set_yticks(_y_pos + _bar_h / 2)
        _ax1.set_yticklabels(_mnames, fontsize=9)
        _ax1.set_xlabel('勝率 / 平均リターン+50 (%)'); _ax1.set_title('モデル別パフォーマンス比較', fontsize=11)
        _ax1.legend(fontsize=8); _ax1.grid(axis='x', alpha=0.3); _ax1.invert_yaxis()

        # 特徴量重要度（最高勝率のツリーモデル）
        if _has_fi:
            _ax2 = axes_ml[1]
            _best_fi = max(feat_imp_dict.keys(), key=lambda m: ml_results[m]['win'])
            _fi = feat_imp_dict[_best_fi]
            _sorted_idx = np.argsort(_fi)
            _ax2.barh(np.array(_ML_FEAT_NAMES)[_sorted_idx], _fi[_sorted_idx], color='#7E57C2')
            _ax2.set_title(f'特徴量重要度  ({_best_fi})', fontsize=11)
            _ax2.set_xlabel('重要度'); _ax2.grid(axis='x', alpha=0.3)

        plt.tight_layout()
        plt.savefig('ml_backtest.png', dpi=130, bbox_inches='tight')
        plt.show()
        print('✓ ML 比較チャートを保存: ml_backtest.png')

        # ベストモデルの判定
        _best_m = max(ml_results.keys(), key=lambda m: ml_results[m]['win'])
        _best_r = ml_results[_best_m]
        print(f'\n  最高勝率モデル: {_best_m}  '
              f'勝率 {_best_r["win"]:.1f}%  平均 {_best_r["mean"]:+.2f}%')
        if _best_r['win'] > 55 and _best_r['mean'] > _all_mean + 0.5:
            print('  ✅ ML モデルがルールベースを明確に上回っています。')
        elif _best_r['win'] > 50:
            print('  🟡 わずかな優位性。追加チューニングや特徴量追加で改善余地あり。')
        else:
            print('  🔴 ML モデルでも明確な優位性は見られません。特徴量やパラメータを見直してください。')
else:
    print('ML バックテストはスキップされました（RUN_ML_BACKTEST = False）')